# Prepare SOC data from FAERS quarterly release

In [1]:
import os, re, time, shutil, pickle
import warnings, pickle, gc, json
import requests
import pandas as pd
import numpy as np
import random
#from tqdm import tqdm, trange
from tqdm.notebook import tqdm, trange
from io import BytesIO
from zipfile import ZipFile
from datetime import datetime
from bs4 import BeautifulSoup
from urllib.request import urlopen

from sentence_transformers import SentenceTransformer
import torch

I0000 00:00:1785209341.296821 1082756 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Skipping import of cpp extensions due to incompatible torch version 2.9.1+cu128 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info
/home/dada/anaconda3/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2274: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a s

In [ ]:
pd.set_option("display.max_columns",  None)

## prepare quarterly string for FAERS since q4 2012

In [ ]:
quarter_string = []
for i in range(2004, 2027):
    for j in ["q1", "q2", "q3", "q4"]:
        quarter_string.append(str(i)+j)

In [ ]:
#get FAERS 2012q4 index
faers_start = quarter_string.index("2012q4")
faers_start

35

In [ ]:
quarter_string[-3:]

['2026q2', '2026q3', '2026q4']

## download quarterly Zip files and unzip to ./faers/folder 

# prepare merge path_in string

In [ ]:
faers_path_in = []
for i in quarter_string[faers_start:-3]:   
#for i in quarter_string[faers_start:]:   
    if "ascii" in os.listdir(f"./faers/{i}"):
        faers_path_in.append(f"./faers/{i}/ascii")
    elif "ASCII" in os.listdir(f"./faers/{i}"):
        faers_path_in.append(f"./faers/{i}/ASCII")

In [ ]:
faers_path_in[-3:]

['./faers/2025q3/ASCII', './faers/2025q4/ASCII', './faers/2026q1/ASCII']

In [ ]:
aers_path_in = []
for i in quarter_string[:faers_start]:   
    if "ascii" in os.listdir(f"./aers_merge/aers/{i}"):
        aers_path_in.append(f"./aers_merge/aers/{i}/ascii")
    elif "ASCII" in os.listdir(f"./aers/{i}"):
        aers_path_in.append(f"./aers_merge/aers/{i}/ASCII")

In [ ]:
aers_path_in[-4:]

['./aers_merge/aers/2011q4/ascii',
 './aers_merge/aers/2012q1/ascii',
 './aers_merge/aers/2012q2/ascii',
 './aers_merge/aers/2012q3/ascii']

In [ ]:
def shiftCol(df):
    df_col = df.columns
    df.reset_index(inplace = True)
    df = df.iloc[:, :len(df_col)]
    #df.columns = [i.lower() for i in df_col]    
    df.columns = df_col    
    return df

# pull all drug names

In [ ]:
def findDrugName(path_in, file_out):
    # merge all files(DEMO, DRUG, REAC, OUTC, INDI) in path_in
    drug_name = []
    if "DRUG" in filename.upper() and "TXT" in filename.upper():
        try:
            drug_df = pd.read_csv(path_in + "/" + filename, sep = "$", low_memory=False)
        except:
            drug_df = pd.read_csv(path_in + "/" + filename, sep = "$", encoding='iso-8859-1', low_memory=False)   
            
        drug_name += list(drug_df.drugname.dropna())        

In [ ]:
path_in = './faers/2023q4/ASCII'
filename = 'DRUG23Q4.txt'

In [ ]:
drug_name = []
if "DRUG" in filename.upper() and "TXT" in filename.upper():
    try:
        drug_df = pd.read_csv(path_in + "/" + filename, sep = "$", low_memory=False)
    except:
        drug_df = pd.read_csv(path_in + "/" + filename, sep = "$", encoding='iso-8859-1', low_memory=False)   
        
    drug_name += list(set(drug_df.drugname.dropna()))

In [ ]:
drug_name[:10] #too complicated

['COVERAM 5 mg/10 mg, comprim?',
 'ESTAZOLAM',
 'CLOBETALSOL',
 'LORAZEPAM EG',
 'FLINTSTONES WITH EXTRA C',
 'GIAVTZCAP',
 'MINT FUROSEMIDE',
 'EXCEDRIN E/S GEL TABLETS 24^S',
 'RYZE',
 'EDARBI CLO']

# create data frame w/o merged caseid

In [ ]:
def normalize_name(x):
    """Normalize FAERS/RxNorm names for exact lookup."""
    if pd.isna(x):
        return None

    x = str(x).lower()
    x = re.sub(r"\s?\((?!\d+\))[^)]*\)", "", x)      # remove parenthetical non-numeric text
    x = re.sub(r"\s?\[.*?\]", "", x)      # remove square-bracketed text
    x = re.sub(r"[^a-z0-9]+", "", x)     # punctuation -> ""
    x = re.sub(r"\s+", " ", x).strip()
    return x or None

In [ ]:
# for filename in os.listdir(path_in):
#     if "DRUG" in filename.upper() and "TXT" in filename.upper():
#         try:
#             df = pd.read_csv(path_in[1] + "/" + filename, sep = "$", low_memory=False)
#         except:
#             df = pd.read_csv(path_in[1] + "/" + filename, sep = "$", encoding='iso-8859-1', low_memory=False)       

In [ ]:
drugs_ = pd.read_csv("./faers/2023q4/ASCII/DRUG23Q4.txt", sep = "$", low_memory=False)

In [ ]:
drugs_.shape

(1920732, 20)

In [ ]:
drugs = drugs_[['primaryid','caseid', 'drugname','route','dose_vbm','dose_amt',
                               'dose_unit','dose_form','dose_freq']]

drugs = drugs[(drugs.drugname.isnull() == False) & (drugs.drugname != "nan")]           

#df['treatment'] = [i.to_dict() for _, i in drug_df.iterrows()]

In [ ]:
drugs.head()

,primaryid,caseid,drugname,route,dose_vbm,dose_amt,dose_unit,dose_form,dose_freq
0,100144838,10014483,ISENTRESS,Transplacental,"400 mg, bid",400.0,MG,Tablet,BID
1,100144838,10014483,ISENTRESS,Transplacental,400 mg,400.0,MG,Tablet,NaN
2,100144838,10014483,ISENTRESS,Transplacental,UNK,NaN,NaN,Tablet,NaN
3,100144838,10014483,ABACAVIR,Transplacental,"300 mg, bid",300.0,MG,Capsule,BID
4,100144838,10014483,ABACAVIR,Transplacental,300 mg,300.0,MG,Capsule,NaN


In [2]:
def normalize_name(x):
    """Normalize FAERS/RxNorm names for exact lookup."""
    if pd.isna(x):
        return None

    x = str(x).lower()
    x = re.sub(r"\s?\((?!\d+\))[^)]*\)", "", x)      # remove parenthetical non-numeric text
    x = re.sub(r"\s?\[.*?\]", "", x)      # remove square-bracketed text
    x = re.sub(r"[^a-z0-9]+", "", x)     # punctuation -> ""
    x = re.sub(r"\s+", " ", x).strip()
    return x or None

In [2]:
import pickle

In [4]:
# map drug name to ingredient with ATC code
all_drugs = pickle.load(open("./RxNorm/all_drugs_ingredient.pkl", "rb"))

In [8]:
all_drugs.tail()

,ingredient_name,norm_name,atc4_name
14170,zongertinib,hernexeos,nan
14171,zonisamide,zonegran,Other antiepileptics
14172,zonisamide,zonisade,Other antiepileptics
14173,zopapogene imadenovec,papzimeos,nan
14174,zuranolone,zurzuvae,Other antidepressants


In [4]:
name_map = dict(zip(all_drugs.norm_name, all_drugs.ingredient_name))

In [5]:
ingredient_atc4 = all_drugs[["ingredient_name", "atc4_name"]].dropna().drop_duplicates()
name_atc4 = dict(zip(ingredient_atc4.ingredient_name, ingredient_atc4.atc4_name))

In [6]:
print(ingredient_atc4.shape)
ingredient_atc4.head()

(2664, 2)


,ingredient_name,atc4_name
0,4-aminobenzoate,nan
1,4-aminosalicylic acid,Aminosalicylic acid and derivatives
2,abacavir,Nucleoside and nucleotide reverse transcriptas...
6,abaloparatide,Parathyroid hormones and analogues
7,abarelix,Other hormone antagonists and related agents


In [2]:
import pickle

In [3]:
#pickle.dump((name_map,name_atc4), open("./RxNorm/name_map_atc4.pkl", "wb"))
name_map,name_atc4 = pickle.load(open("./RxNorm/name_map_atc4.pkl", "rb"))

In [ ]:
# for every drug, if it maps an ingredient_name, if ingredient_name has atc4 name, get it
drugs.drugname[-5:]

1920727    TRAMADOL HYDROCHLORIDE
1920728                  VOLTAREN
1920729                  NAPROXEN
1920730                PREDNISONE
1920731              METHOTREXATE
Name: drugname, dtype: object

In [ ]:
drugs["drugname"] = [normalize_name(x) for x in drugs.drugname]

In [ ]:
def normDrugName(drugname_lst):

    drugname_ = [normalize_name(x) for x in drugname_lst]
    drugname_ingredient = [name_map[k] if k in name_map.keys() else k for k in drugname_]
    durgname_atc4 = [ingredient_atc4[k] if k in ingredient_atc4.keys() else k for k in drugname_ingredient]

    return drugname_ingredient, durgname_atc4

In [ ]:
def cleanCol(df, col):
    #['drugname','route','dose_vbm', "dose_amt","dose_unit","dose_form","dose_freq"]:        

    for i in col: 
        if df[i].dtype == "float64":
            df[i] = df[i].astype(str)
        else:
            df[i] = np.where((df[i].isnull()) | (df[i] == "UNK"), "nan", df[i].str.lower())

    return df

In [ ]:
def preProcess(drugs, data_type):
    '''
    map drug name with ingredient and ATC4 level info
    '''
    
    global name_map, name_atc4

    if data_type == "FAERS":    
        drugs["ingredient"] = [name_map[k] if k in name_map.keys() else k for k in drugs.drugname]
        drugs["atc4"] = [name_atc4[k].lower() if k in name_atc4.keys() else k for k in drugs.ingredient]
        drugs = cleanCol(drugs, ['route','dose_vbm','dose_amt', 'dose_unit','dose_form','dose_freq'])
        drugs['dose'] = drugs[['dose_amt', 'dose_unit','dose_form','dose_freq']].apply(lambda x: " ".join(x.astype(str)), axis=1)
        drugs.drop(["dose_vbm","dose_amt","dose_unit","dose_form","dose_freq"], axis = 1, inplace = True)

        # get dose rank, the coutn of nan in dose 
        drugs['dose_rank'] = [s.split(" ").count("nan") for s in drugs.dose]
        drugs = (drugs.sort_values(['primaryid', 'caseid', 'drugname', 'dose_rank'], ascending = False)    
        .drop(columns='dose_rank')
        .reset_index(drop=True))
    elif data_type == "AERS":
        drugs["ingredient"] = [name_map[k] if k in name_map.keys() else k for k in drugs.drugname]
        drugs["atc4"] = [name_atc4[k].lower() if k in name_atc4.keys() else k for k in drugs.ingredient]
        drugs = cleanCol(drugs, ['route','dose'])
    else:
        print("Select ADR data type.")        
    
    return drugs


# define FARES merge func

In [ ]:
def mergeFAERS(path_in, path_out):

    '''for every unique 'caseid', only the last 'primaryid' will be used'''
    
    # merge all files(DEMO, DRUG, REAC, OUTC, INDI) in path_in
    for filename in os.listdir(path_in):

        file_out = path_in.split("/")[2]
        
        if "DEMO" in filename.upper() and "TXT" in filename.upper():
            try:
                demo_df = pd.read_csv(path_in + "/" + filename, sep = "$", low_memory=False)
            except:
                demo_df = pd.read_csv(path_in + "/" + filename, sep = "$", encoding='iso-8859-1', low_memory=False)   
                
            if "sex" in demo_df.columns:
                demo_df.rename(columns = {"sex":'gndr_cod'}, inplace = True)
            #demo_df = demo_df[['primaryid','caseid','age','age_cod','gndr_cod','wt','wt_cod','occp_cod','reporter_country','occr_country']]
            demo_df = demo_df[['fda_dt','rept_cod', 'primaryid','caseid','age','age_cod','gndr_cod','wt','wt_cod']]
            demo_df = demo_df[(demo_df.wt.isnull() == False) & (demo_df.age.isnull() == False)]
            #demo_df.drop('caseid', axis = 1, inplace = True)
            
        if "DRUG" in filename.upper() and "TXT" in filename.upper():
            try:
                drug_df = pd.read_csv(path_in + "/" + filename, sep = "$", low_memory=False)
            except:
                drug_df = pd.read_csv(path_in + "/" + filename, sep = "$", encoding='iso-8859-1', low_memory=False)   
            
            # drugname	route	dose_vbm	dose_amt	dose_unit	dose_form	dose_freq	dose
            drug_df = drug_df[['primaryid', "caseid", 'drugname','route','dose_vbm','dose_amt',
                               'dose_unit','dose_form','dose_freq']]

            #lower col values
            drug_df = cleanCol(drug_df, ['drugname','route','dose_vbm','dose_amt', 'dose_unit','dose_form','dose_freq'])
            drug_df = drug_df[(drug_df.drugname != "nan") & (drug_df.drugname != "unk")]   

            # get ingredient and atc4
            drug_df = preProcess(drug_df, "FAERS")            
            drug_df['treatment'] = [re.sub(r"[|+|\\+]", "", json.dumps(i.to_dict())) for _, i in
                      drug_df[['drugname','ingredient', 'atc4', 'route','dose']].iterrows()]
            #group treatment by each report and separate by "; " without duplicates
            drug_df = drug_df.groupby(['primaryid', 'caseid']).treatment.apply(lambda x: "; ".join(set(x.astype(str)))).reset_index()
            
        if "OUTC" in filename.upper() and "TXT" in filename.upper():            
            outc_df = pd.read_csv(path_in + "/" + filename, sep = "$", low_memory=False)                   
            outc_df = outc_df.groupby(['primaryid', 'caseid'])[outc_df.columns[2]].apply("; ".join).reset_index()
            #outc_df = outc_df.groupby(['primaryid', 'caseid'])[outc_df.columns[2]].last()

        if "INDI" in filename.upper() and "TXT" in filename.upper():            
            indi_df = pd.read_csv(path_in + "/" + filename, sep = "$", low_memory=False) 
            indi_df = indi_df.groupby(['primaryid', 'caseid']).indi_pt.apply(lambda x: "; ".join(x.astype(str).str.lower())).reset_index()
            #indi_df = indi_df.groupby(['primaryid', 'caseid']).indi_pt.last()
            
        if "REAC" in filename.upper() and "TXT" in filename.upper():            
            reac_df = pd.read_csv(path_in + "/" + filename, sep = "$",low_memory=False)            
            reac_df = reac_df.groupby(['primaryid', 'caseid']).pt.apply(lambda x: "; ".join(x.astype(str).str.lower())).reset_index()
            #reac_df = reac_df.groupby(['primaryid', 'caseid']).pt.last()

    #merge files based on primary report id and case id
    quarter_df = pd.merge(demo_df, drug_df[['primaryid', 'caseid', 'treatment']], 
                          on=['primaryid', 'caseid'], how='inner')  
    quarter_df = pd.merge(quarter_df, outc_df, on=['primaryid', 'caseid'], how = "left")  
    quarter_df = pd.merge(quarter_df, indi_df, on=['primaryid', 'caseid']) # how='inner'
    quarter_df = pd.merge(quarter_df, reac_df, on=['primaryid', 'caseid']) # how='inner'
        
    print(f"The shape of {file_out} is {quarter_df.shape}")

    if os.path.exists(path_out):
        pickle.dump(quarter_df, open(f"{path_out}/{file_out}.pkl", "wb"))
    else:
        !mkdir {path_out}
        pickle.dump(quarter_df, open(f"{path_out}/{file_out}.pkl", "wb"))    
    #pickle.dump(quarter_df, open(f"./aers_merge/{file_out}.pkl", "wb"))

In [ ]:
from functools import partial
from itertools import repeat
from multiprocessing import Pool, freeze_support

In [ ]:
# %%time 

# with Pool() as pool:
    
#     list(tqdm(pool.imap(partial(mergeFAERS, path_out= "./faers_new/"), faers_path_in), 
#         total=len(faers_path_in), desc="Processing")) 

# pool.close() #34K to 70K

In [ ]:
q126 = pickle.load(open("/home/dada/Barn/GQ/ADR/faers_new/2026q1.pkl", "rb"))

In [ ]:
q126.head(2)

,fda_dt,rept_cod,primaryid,caseid,age,age_cod,gndr_cod,wt,wt_cod,treatment,outc_cod,indi_pt,pt
0,20260127,EXP,1005762126,10057621,57.0,YR,M,53.000,KG,"{""drugname"": ""xolair"", ""ingredient"": ""omalizum...",HO,asthma; product used for unknown indication; p...,fall; nasal polyps; dizziness; paraesthesia; h...
1,20260303,EXP,101092286,10109228,58.0,YR,F,96.599,KG,"{""drugname"": ""lomitapide"", ""ingredient"": ""lomi...",OT,type iia hyperlipidaemia; carotid artery disea...,aspartate aminotransferase increased; abdomina...


In [ ]:
print(q126.treatment[4].replace(";", "\n"))

{"drugname": "medrol", "ingredient": "methylprednisolone", "atc4": "glucocorticoids, corticosteroids, combinations for treatment of acne, corticosteroids, weak (group i)", "route": "nan", "dose": "nan nan tablet qd"}
 {"drugname": "xolair", "ingredient": "omalizumab", "atc4": "other systemic drugs for obstructive airway diseases", "route": "nan", "dose": "nan nan nan nan"}
 {"drugname": "medrol", "ingredient": "methylprednisolone", "atc4": "glucocorticoids, corticosteroids, combinations for treatment of acne, corticosteroids, weak (group i)", "route": "nan", "dose": "nan nan tablet /wk"}
 {"drugname": "glycopyrrolate", "ingredient": "glycopyrrolate", "atc4": "glycopyrrolate", "route": "nan", "dose": "nan nan tablet nan"}
 {"drugname": "symbicort", "ingredient": "formoterol", "atc4": "selective beta-2-adrenoreceptor agonists, selective beta-2-adrenoreceptor agonists", "route": "nan", "dose": "nan nan nan nan"}
 {"drugname": "xolair", "ingredient": "omalizumab", "atc4": "other systemic d

# Prep AERS quarterly data before 2012Q4

In [ ]:
drug_df = pd.read_csv("/home/dada/Barn/GQ/ADR/aers_merge/aers/2012q2/ascii/DRUG12Q2.TXT", 
                      sep = "$", encoding='iso-8859-1', low_memory=False)   

drug_df = shiftCol(drug_df)            
drug_df.rename(columns = {"DOSE_VBM":"DOSE"}, inplace = True)
drug_df = drug_df[['ISR', 'DRUGNAME','ROUTE','DOSE']] #subset drug_df
drug_df.rename(columns = {'DRUGNAME':"drugname",'ROUTE':"route",'DOSE':"dose"}, inplace = True)
drug_df = drug_df.loc[(drug_df.drugname.isnull() == False) & (drug_df.drugname != "nan"),:] #filter drug_df

drug_df["drugname"] = drug_df.drugname.str.lower()
          
# get ingredient and atc4
drug_df = preProcess(drug_df, "AERS")     
drug_df['TREATMENT'] = [re.sub(r"[|+|\\+]", "", json.dumps(i.to_dict())) for _, i 
        in drug_df[['drugname','ingredient', 'atc4','route','dose']].iterrows()]       

In [ ]:
drug_df['TREATMENT'] = [re.sub(r"[|+|\\+]", "", json.dumps(i.to_dict())) for _, i 
        in drug_df[['drugname','ingredient', 'atc4','route','dose']].iterrows()]

In [ ]:
drug_df.head()

,ISR,drugname,route,dose,ingredient,atc4,TREATMENT
0,8017085,radiation therapy,nan,nan,radiation therapy,radiation therapy,"{""drugname"": ""radiation therapy"", ""ingredient""..."
1,8017085,decadron,nan,10 mg,dexamethasone,"corticosteroids, moderately potent, other comb...","{""drugname"": ""decadron"", ""ingredient"": ""dexame..."
2,8017085,rocephin,intravenous,4 gm,ceftriaxone,third-generation cephalosporins,"{""drugname"": ""rocephin"", ""ingredient"": ""ceftri..."
3,8017085,morphine sulfate,nan,30 mg / prn,morphine sulfate,morphine sulfate,"{""drugname"": ""morphine sulfate"", ""ingredient"":..."
4,8017085,vitamin tab,nan,nan,vitamin tab,vitamin tab,"{""drugname"": ""vitamin tab"", ""ingredient"": ""vit..."


In [ ]:
def mergeAERS(path_in, path_out):

    '''group by unique 'ISR' '''
    #print(f"Processing {path_in}\n")
    # merge all files(DEMO, DRUG, REAC, OUTC, INDI) in path_in
    for filename in os.listdir(path_in):

        file_out = path_in.split("/")[3]
        
        if "DEMO" in filename.upper() and "TXT" in filename.upper():
            try:
                demo_df = pd.read_csv(path_in + "/" + filename, sep = "$", low_memory=False)
            except:
                demo_df = pd.read_csv(path_in + "/" + filename, sep = "$", encoding='iso-8859-1', low_memory=False)   
            
            demo_df = shiftCol(demo_df)
            if "SEX" in demo_df.columns:
                demo_df.rename(columns = {"SEX":'GNDR_COD'}, inplace = True)
            #demo_df = demo_df[['primaryid','caseid','age','age_cod','gndr_cod','wt','wt_cod','occp_cod','reporter_country','occr_country']]
            demo_df = demo_df[['FDA_DT','REPT_COD',"ISR",'AGE','AGE_COD','GNDR_COD','WT','WT_COD']]
            demo_df = demo_df[(demo_df.WT.isnull() == False) & (demo_df.AGE.isnull() == False)]
            #demo_df.drop('caseid', axis = 1, inplace = True)
            
        if "DRUG" in filename.upper() and "TXT" in filename.upper():
            try:
                drug_df = pd.read_csv(path_in + "/" + filename, sep = "$", low_memory=False)
            except:
                drug_df = pd.read_csv(path_in + "/" + filename, sep = "$", encoding='iso-8859-1', low_memory=False)   

            drug_df = shiftCol(drug_df)
            drug_df.rename(columns = {"DOSE_VBM":"DOSE"}, inplace = True)
            drug_df = drug_df[['ISR', 'DRUGNAME','ROUTE','DOSE']] #subset drug_df
            drug_df.rename(columns = {'DRUGNAME':"drugname",'ROUTE':"route",'DOSE':"dose"}, inplace = True)
            drug_df = drug_df.loc[(drug_df.drugname.isnull() == False) & (drug_df.drugname != "nan"),:] #filter drug_df
            
            drug_df["drugname"] = drug_df.drugname.str.lower()
            
            # get ingredient and atc4
            drug_df = preProcess(drug_df, "AERS")     
            drug_df['TREATMENT'] = [re.sub(r"[|+|\\+]", "", json.dumps(i.to_dict())) for _, i in drug_df[['drugname','ingredient', 'atc4','route','dose']].iterrows()] 
            #group treatment by each report and separate by "; " without duplicates
            drug_df = drug_df.groupby(['ISR']).TREATMENT.apply(lambda x: "; ".join(set(x.astype(str)))).reset_index()
            
        if "OUTC" in filename.upper() and "TXT" in filename.upper():            
            outc_df = pd.read_csv(path_in + "/" + filename, sep = "$", low_memory=False)    
            outc_df = shiftCol(outc_df)
            outc_df = outc_df.groupby(['ISR'])[outc_df.columns[1]].apply("; ".join).reset_index()
            #outc_df = outc_df.groupby(['primaryid', 'caseid'])[outc_df.columns[2]].last()

        if "INDI" in filename.upper() and "TXT" in filename.upper():            
            indi_df = pd.read_csv(path_in + "/" + filename, sep = "$", low_memory=False) 
            #indi_df = shiftCol(indi_df)
            indi_df = indi_df.groupby(['ISR']).INDI_PT.apply(lambda x: "; ".join(x.astype(str))).reset_index()
            #indi_df = indi_df.groupby(['primaryid', 'caseid']).indi_pt.last()
            
        if "REAC" in filename.upper() and "TXT" in filename.upper():            
            reac_df = pd.read_csv(path_in + "/" + filename, sep = "$",low_memory=False)      
            reac_df = shiftCol(reac_df)
            reac_df = reac_df.groupby(['ISR']).PT.apply(lambda x: "; ".join(x.astype(str))).reset_index()
            #reac_df = reac_df.groupby(['primaryid', 'caseid']).pt.last()

    #merge files based on primary report id and case id
    quarter_df = pd.merge(demo_df, drug_df[['ISR', 'TREATMENT']], on=['ISR'], how='inner')  
    #print(quarter_df.shape)
    quarter_df = pd.merge(quarter_df, outc_df, on=['ISR'], how = "left")  
    #print(quarter_df.shape)
    quarter_df = pd.merge(quarter_df, indi_df, on=['ISR']) # how='inner'
    #print(quarter_df.shape)
    quarter_df = pd.merge(quarter_df, reac_df, on=['ISR']) # how='inner'
        
    print(f"The shape of {file_out} is {quarter_df.shape}")

    if os.path.exists(path_out):
        pickle.dump(quarter_df, open(f"{path_out}/{file_out}.pkl", "wb"))
    else:
        !mkdir {path_out}
        pickle.dump(quarter_df, open(f"{path_out}/{file_out}.pkl", "wb"))  

In [ ]:
%%time 

with Pool() as pool:    
    list(tqdm(pool.imap(partial(mergeAERS, path_out= "./aers_new"), aers_path_in), 
        total=len(aers_path_in), desc="Processing")) 

pool.close() #17K to 38K

Processing:   0%|          | 0/35 [00:00<?, ?it/s]

The shape of 2004q1 is (18887, 12)


Processing:   3%|▎         | 1/35 [00:36<20:40, 36.49s/it]

The shape of 2004q2 is (16943, 12)


Processing:   6%|▌         | 2/35 [00:37<08:29, 15.45s/it]

...

The shape of 2012q1 is (52145, 12)


Processing:  94%|█████████▍| 33/35 [03:46<00:11,  5.58s/it]

The shape of 2012q2 is (50730, 12)


Processing: 100%|██████████| 35/35 [03:48<00:00,  6.53s/it]


CPU times: user 113 ms, sys: 2.27 s, total: 2.38 s
Wall time: 3min 51s


# consolidate aers pkls into a single pickle

In [35]:
pkl_lst = os.listdir('./aers_new/')
pkl_lst = sorted(pkl_lst)
pkl_lst[-1]

'2012q3.pkl'

In [51]:
%%time 

aers_df = pd.DataFrame()
for i in tqdm(pkl_lst):
    pkl_i = pickle.load(open(f"./aers_new/{i}", "rb"))
    if "OUTC_COD" in pkl_i.columns:        
        '''rename outc_cod to outc_code before appending'''
        pkl_i.rename(columns = {"OUTC_COD": "OUTC_CODE"}, inplace = True)    
        
    aers_df = pd.concat([aers_df, pkl_i])

  0%|          | 0/35 [00:00<?, ?it/s]

CPU times: user 1.92 s, sys: 228 ms, total: 2.15 s
Wall time: 2.14 s


In [52]:
aers_df.reset_index(drop = True, inplace = True)

In [53]:
print(aers_df.TREATMENT[0].replace("; ", "\n"))

{"drugname": "analgesics", "ingredient": "analgesics", "atc4": "analgesics", "route": "nan", "dose": "nan"}
{"drugname": "zithromax", "ingredient": "azithromycin", "atc4": "antibiotics, macrolides", "route": "oral", "dose": "250 mg (daily), oral"}
{"drugname": "acetazolamide", "ingredient": "acetazolamide", "atc4": "carbonic anhydrase inhibitors", "route": "nan", "dose": "nan"}


In [54]:
aers_df.columns = [i.lower() for i in aers_df.columns] #lower case col names

In [56]:
aers_df.head()

,fda_dt,rept_cod,isr,age,age_cod,gndr_cod,wt,wt_cod,treatment,outc_code,indi_pt,pt
0,20031229,EXP,4261678,52.0,YR,F,200.0,LBS,"{""drugname"": ""analgesics"", ""ingredient"": ""anal...",HO; OT,SKIN DISORDER; BENIGN INTRACRANIAL HYPERTENSIO...,BRONCHITIS; DIARRHOEA; DISORIENTATION; DRUG IN...
1,20040102,EXP,4261826,71.0,YR,M,110.0,KG,"{""drugname"": ""beta blocking agents"", ""ingredie...",HO,OESOPHAGEAL CARCINOMA; HYPERTENSION; ATRIAL FI...,CUTANEOUS VASCULITIS; FEBRILE NEUTROPENIA; REN...
2,20040102,EXP,4261829,51.0,YR,F,82.1,KG,"{""drugname"": ""coumadin"", ""ingredient"": ""warfar...",DE; HO,ANAL CANCER; ANAL CANCER; PAIN,ANAL CANCER; COAGULOPATHY; DYSPNOEA; GASTROINT...
3,20040102,EXP,4261860,49.0,YR,M,35.0,KG,"{""drugname"": ""calcium carbonate"", ""ingredient""...",DE; HO,RENAL TRANSPLANT; RENAL TRANSPLANT; RENAL TRAN...,ACUTE PULMONARY OEDEMA; ACUTE RESPIRATORY DIST...
4,20040102,EXP,4261872,72.0,YR,M,63.0,KG,"{""drugname"": ""bufferin"", ""ingredient"": ""acetyl...",DS; HO,HYPERTENSION; AGE INDETERMINATE MYOCARDIAL INF...,ATRIOVENTRICULAR BLOCK; ATRIOVENTRICULAR BLOCK...


# create ARES `inst` string

In [57]:
aers_df.head(2)

,fda_dt,rept_cod,isr,age,age_cod,gndr_cod,wt,wt_cod,treatment,outc_code,indi_pt,pt
0,20031229,EXP,4261678,52.0,YR,F,200.0,LBS,"{""drugname"": ""analgesics"", ""ingredient"": ""anal...",HO; OT,SKIN DISORDER; BENIGN INTRACRANIAL HYPERTENSIO...,BRONCHITIS; DIARRHOEA; DISORIENTATION; DRUG IN...
1,20040102,EXP,4261826,71.0,YR,M,110.0,KG,"{""drugname"": ""beta blocking agents"", ""ingredie...",HO,OESOPHAGEAL CARCINOMA; HYPERTENSION; ATRIAL FI...,CUTANEOUS VASCULITIS; FEBRILE NEUTROPENIA; REN...


In [58]:
aers_df['yr_qtr'] = aers_df.fda_dt = pd.to_datetime(aers_df.fda_dt.astype(str), format='%Y%m%d')
aers_df["yr_qtr"] = aers_df.fda_dt.dt.year.astype(str) + "Q" + aers_df.fda_dt.dt.quarter.astype(str)

In [59]:
%%time

aers_df['patient'] = [re.sub(r"[|+|\\+]", "", json.dumps(row.to_dict())) 
                        for _, row in aers_df[['age','age_cod','gndr_cod','wt','wt_cod']].astype(str).iterrows()]
aers_df.drop(['age','age_cod','gndr_cod','wt','wt_cod'], axis = 1, inplace = True)

# add yr_qtr to inst string
aers_df['inst'] = [re.sub(r"[|+|\\+]", "", json.dumps(row.to_dict()))
                      for _, row in aers_df[['patient','treatment','indi_pt', 'yr_qtr']].astype(str).iterrows()]
aers_df.drop(['patient','treatment','indi_pt'], axis = 1, inplace = True)
aers_df.pt = [i.lower() for i in aers_df.pt]

CPU times: user 3min 11s, sys: 3.31 s, total: 3min 14s
Wall time: 3min 13s


In [65]:
aers_df.head(2)

,fda_dt,rept_cod,isr,outc_code,pt,yr_qtr,inst
0,2003-12-29,EXP,4261678,HO; OT,bronchitis; diarrhoea; disorientation; drug in...,2003Q4,"{""patient"": ""{""age"": ""52.0"", ""age_cod"": ""yr"", ..."
1,2004-01-02,EXP,4261826,HO,cutaneous vasculitis; febrile neutropenia; ren...,2004Q1,"{""patient"": ""{""age"": ""71.0"", ""age_cod"": ""yr"", ..."


In [ ]:
aers_df.inst[10]

'{"patient": "{"age": "49.0", "age_cod": "yr", "gndr_cod": "m", "wt": "85.0", "wt_cod": "kg"}", "treatment": "{"drugname": "malarone", "ingredient": "proguanil", "atc4": "biguanides", "route": "oral", "dose": "1tab per day"}", "indi_pt": "malaria prophylaxis"}'

In [61]:
aers_df.inst = [i.lower() for i in aers_df.inst]

In [62]:
aers_df.pt = [i.lower() for i in aers_df.pt]

In [67]:
aers_df.inst[0]

'{"patient": "{"age": "52.0", "age_cod": "yr", "gndr_cod": "f", "wt": "200.0", "wt_cod": "lbs"}", "treatment": "{"drugname": "analgesics", "ingredient": "analgesics", "atc4": "analgesics", "route": "nan", "dose": "nan"}; {"drugname": "zithromax", "ingredient": "azithromycin", "atc4": "antibiotics, macrolides", "route": "oral", "dose": "250 mg (daily), oral"}; {"drugname": "acetazolamide", "ingredient": "acetazolamide", "atc4": "carbonic anhydrase inhibitors", "route": "nan", "dose": "nan"}", "indi_pt": "skin disorder; benign intracranial hypertension; headache", "yr_qtr": "2003q4"}'

In [64]:
pickle.dump(aers_df, open("adr_up2_2012_q3_new.pkl", "wb"))

In [77]:
aers_df = pickle.load(open("adr_up2_2012_q3_new.pkl", "rb"))

In [79]:
4498178 in list(aers_df.isr)

True

In [83]:
aers_df.iloc[aers_df.isr == 4498455]

,fda_dt,rept_cod,isr,outc_code,pt,yr_qtr,inst
62094,2004-11-05,EXP,4498455,DE; HO,hepatitis fulminant,2004Q4,"{""patient"": ""{""age"": ""71.0"", ""age_cod"": ""yr"", ..."


In [84]:
aers_df.iloc[aers_df.inst == aers_df.inst[62094]]

,fda_dt,rept_cod,isr,outc_code,pt,yr_qtr,inst
62094,2004-11-05,EXP,4498455,DE; HO,hepatitis fulminant,2004Q4,"{""patient"": ""{""age"": ""71.0"", ""age_cod"": ""yr"", ..."
68121,2004-12-08,EXP,4523002,DE; HO; OT,arrhythmia; blood albumin decreased; coagulopa...,2004Q4,"{""patient"": ""{""age"": ""71.0"", ""age_cod"": ""yr"", ..."


# consolidate fares pkls into a single pickle

In [68]:
pkl_lst = os.listdir('./faers_new/')
pkl_lst = sorted(pkl_lst)
pkl_lst[-1]

'2026q1.pkl'

In [69]:
%%time 

append_df = pd.DataFrame()
for i in tqdm(pkl_lst):
    pkl_i = pickle.load(open(f"./faers_new/{i}", "rb"))
    if "outc_cod" in pkl_i.columns:        
        '''rename outc_cod to outc_code before appending'''
        pkl_i.rename(columns = {"outc_cod": "outc_code"}, inplace = True)    
        
    append_df = pd.concat([append_df, pkl_i])

  0%|          | 0/54 [00:00<?, ?it/s]

CPU times: user 10.2 s, sys: 1.71 s, total: 11.9 s
Wall time: 12 s


In [70]:
append_df.shape #2.96M 

(2969353, 13)

In [71]:
append_df.outc_code = append_df.outc_code.astype(str)

# combine faers and aers df

In [72]:
append_df.tail(2)

,fda_dt,rept_cod,primaryid,caseid,age,age_cod,gndr_cod,wt,wt_cod,treatment,outc_code,indi_pt,pt
55485,20260313,EXP,963050426,9630504,51.0,YR,F,79.4,KG,"{""drugname"": ""rituxan"", ""ingredient"": ""rituxim...",HO; OT,rheumatoid arthritis; lymphoma; premedication;...,cough; cardiac disorder; dizziness; influenza;...
55486,20260122,EXP,96997894,9699789,61.0,YR,F,62.0,KG,"{""drugname"": ""gliclazide"", ""ingredient"": ""glic...",HO,breast cancer; breast cancer; breast cancer; b...,diarrhoea; nausea; dysuria; diarrhoea; vomiting


In [73]:
append_df['yr_qtr'] = append_df.fda_dt = pd.to_datetime(append_df.fda_dt.astype(str), format='%Y%m%d')
append_df["yr_qtr"] = append_df.fda_dt.dt.year.astype(str) + "Q" + append_df.fda_dt.dt.quarter.astype(str)

# create FARES inst string

In [74]:
%%time

append_df['patient'] = [re.sub(r"[|+|\\+]", "", json.dumps(row.to_dict()))
                        for _, row in append_df[['age','age_cod','gndr_cod','wt','wt_cod']].astype(str).iterrows()]
append_df.drop(['age','age_cod','gndr_cod','wt','wt_cod'], axis = 1, inplace = True)
append_df['inst'] = [re.sub(r"[|+|\\+]", "", json.dumps(row.to_dict()))
                      for _, row in append_df[['patient','treatment','indi_pt', 'yr_qtr']].astype(str).iterrows()]
append_df.drop(['patient','treatment','indi_pt'], axis = 1, inplace = True)
append_df.pt = [i.lower() for i in append_df.pt]
append_df.inst = [i.lower() for i in append_df.inst]

CPU times: user 9min 19s, sys: 18.4 s, total: 9min 37s
Wall time: 9min 34s


In [75]:
append_df.reset_index(inplace = True, drop = True)

In [76]:
append_df.inst[10]

'{"patient": "{"age": "53", "age_cod": nan, "gndr_cod": "m", "wt": "52.15", "wt_cod": "kg"}", "treatment": "{"drugname": "glivec", "ingredient": "glivec", "atc4": "glivec", "route": "oral", "dose": "600 mg nan qd"}", "indi_pt": "chronic myeloid leukaemia", "yr_qtr": "2012q3"}'

In [77]:
append_df.head(2)

,fda_dt,rept_cod,primaryid,caseid,outc_code,pt,yr_qtr,inst
0,2012-11-20,EXP,37831703,3783170,HO,methaemoglobinaemia; overdose,2012Q4,"{""patient"": ""{""age"": ""3"", ""age_cod"": ""yr"", ""gn..."
1,2012-09-20,EXP,37883263,3788326,OT,hot flush; thermal burn,2012Q3,"{""patient"": ""{""age"": ""49"", ""age_cod"": ""yr"", ""g..."


In [78]:
pickle.dump(append_df, open("adr_up2_2026_q1.pkl", "wb"))

In [74]:
# import pickle
append_df = pickle.load(open("adr_up2_2026_q1.pkl", "rb"))

In [76]:
type(append_df.primaryid[0])

numpy.int64

In [80]:
4498178 in list(append_df.primaryid)

False

# combine 2 subsets

In [85]:
# sort by fda_dt drop duplicater primaryid
append_df = append_df.sort_values(by="fda_dt")
append_df = append_df.drop_duplicates(subset = ["primaryid"], keep='last')

In [86]:
len(append_df.primaryid.unique()) #2969119

2969119

In [87]:
append_df.drop("caseid", axis = 1, inplace = True)
append_df.rename(columns = {"primaryid":"caseid"}, inplace = True)

In [88]:
aers_df.rename(columns = {"isr":"caseid"}, inplace = True)

In [89]:
aers_df.head(2)

,fda_dt,rept_cod,caseid,outc_code,pt,yr_qtr,inst
0,2003-12-29,EXP,4261678,HO; OT,bronchitis; diarrhoea; disorientation; drug in...,2003Q4,"{""patient"": ""{""age"": ""52.0"", ""age_cod"": ""yr"", ..."
1,2004-01-02,EXP,4261826,HO,cutaneous vasculitis; febrile neutropenia; ren...,2004Q1,"{""patient"": ""{""age"": ""71.0"", ""age_cod"": ""yr"", ..."


In [ ]:
adr_full = pd.concat([append_df, aers_df])

In [ ]:
adr_full.shape

(3975220, 7)

In [ ]:
adr_full.sort_values(by="fda_dt", inplace=True)
adr_full.drop_duplicates(subset = ["caseid"], keep='last', inplace = True)

In [ ]:
print(adr_full.shape)
adr_full.head() 

(3973842, 7)


,fda_dt,rept_cod,caseid,outc_code,pt,yr_qtr,inst
1703653,1997-12-12,EXP,31231172,DE; HO,arterial thrombosis; drug interaction; haemorr...,1997Q4,"{""patient"": ""{""age"": ""59.0"", ""age_cod"": ""yr"", ..."
1703657,1998-07-08,PER,32427281,OT,anaemia; dermatitis; ecchymosis,1998Q3,"{""patient"": ""{""age"": ""46.0"", ""age_cod"": ""yr"", ..."
1703672,1998-12-18,EXP,37847331,RI,blood pressure decreased; rash erythematous,1998Q4,"{""patient"": ""{""age"": ""72.0"", ""age_cod"": ""yr"", ..."
1703654,1999-02-02,EXP,32030921,LT; DE,blister; dermatitis; lip disorder; mucosal ero...,1999Q1,"{""patient"": ""{""age"": ""34.0"", ""age_cod"": ""yr"", ..."
1703655,1999-02-18,EXP,32138152,LT; DE,conjunctivitis; mouth ulceration; oral mucosal...,1999Q1,"{""patient"": ""{""age"": ""79.0"", ""age_cod"": ""yr"", ..."


In [ ]:
adr_full.reset_index(drop = True, inplace = True)

In [ ]:
adr_full.drop(["fda_dt","rept_cod"], axis = 1, inplace = True)

In [27]:
#pickle.dump(adr_all, open("adr_full_up2_2026_q1.pkl", "wb"))
adr_full = pickle.load(open("adr_full_up2_2026_q1.pkl", "rb"))

In [ ]:
# drop duplicated instances
adr_all = adr_full.drop_duplicates(subset = ["inst"], keep='last') #there are duplicated "caseid"

In [101]:
adr_all.shape

(3866480, 5)

In [102]:
adr_all.reset_index(drop = True, inplace = True)

# prepare training data for finetuning BGE-m3

In [11]:
adr_all.tail(2)

,caseid,outc_code,pt,yr_qtr,inst
3866478,243677083,OT,limb injury; hepatic steatosis; dyslipidaemia;...,2026Q1,"{""patient"": ""{""age"": ""82.0"", ""age_cod"": ""yr"", ..."
3866479,265225022,OT,dyspnoea; sinusitis; night sweats; headache; b...,2026Q1,"{""patient"": ""{""age"": ""69.0"", ""age_cod"": ""yr"", ..."


In [ ]:
adr_all.drop(["fda_dt","rept_cod"], axis = 1, inplace = True)

In [7]:
adr_all.shape

(3866480, 5)

# Save adr all for SOC project

In [4]:
# pickle.dump(adr_all, open("adr_all_new_up2_26q1.pkl", "wb"))
adr_all = pickle.load(open("adr_all_new_up2_26q1.pkl", "rb")) #concat tst and trn in order

In [5]:
print(adr_all.shape)
adr_all.head(2)

(3866480, 5)


,caseid,outc_code,pt,yr_qtr,inst
0,31231172,DE; HO,arterial thrombosis; drug interaction; haemorr...,1997Q4,"{""patient"": ""{""age"": ""59.0"", ""age_cod"": ""yr"", ..."
1,32427281,OT,anaemia; dermatitis; ecchymosis,1998Q3,"{""patient"": ""{""age"": ""46.0"", ""age_cod"": ""yr"", ..."


In [6]:
adr_all.inst[0]

'{"patient": "{"age": "59.0", "age_cod": "yr", "gndr_cod": "f", "wt": "150.0", "wt_cod": "lbs"}", "treatment": "{"drugname": "photofrin", "ingredient": "dihematoporphyrin ether", "atc4": "nan", "route": "intravenous (not otherwise specified)", "dose": "nan nan nan nan"}; {"drugname": "calcium", "ingredient": "calcium", "atc4": "calcium", "route": "nan", "dose": "nan nan nan nan"}; {"drugname": "lasix", "ingredient": "furosemide", "atc4": "sulfonamides, plain", "route": "nan", "dose": "nan nan nan nan"}; {"drugname": "coumadin", "ingredient": "warfarin", "atc4": "vitamin k antagonists", "route": "nan", "dose": "nan nan nan nan"}; {"drugname": "prevacid", "ingredient": "lansoprazole", "atc4": "proton pump inhibitors", "route": "nan", "dose": "nan nan nan nan"}; {"drugname": "ferrous sulfate.", "ingredient": "ferrous sulfate.", "atc4": "ferrous sulfate.", "route": "nan", "dose": "nan nan nan nan"}; {"drugname": "dilantin", "ingredient": "phenytoin", "atc4": "hydantoin derivatives", "route

In [90]:
trn_idx = adr_all.index[adr_all.yr_qtr <= "2024Q4"].tolist()
tst_idx = adr_all.index[adr_all.yr_qtr > "2024Q4"].tolist()

In [9]:
print(len(tst_idx) + len(trn_idx)) # the same dimension as adr_trn and adr_tst

3866480


In [ ]:
#pickle.dump((trn_idx, tst_idx), open("adr_trn_tst_idx.pkl", "wb"))
(trn_idx, tst_idx) = pickle.load(open("adr_trn_tst_idx.pkl", "rb"))

In [40]:
len(trn_idx) + len(tst_idx)

3866480

In [9]:
#output isnt_all for embedding
pickle.dump(adr_all.inst.tolist(), open("adr_inst_lst.pkl", "wb"))

# use live-API/LLM enrichment tiers

In [10]:
ingredient_atc4_map = pickle.load(open("drugname_ingredient_atc4_map.pkl", "rb"))

In [12]:
len(ingredient_atc4_map)

474340

In [15]:
ingredient_atc4_map[list(ingredient_atc4_map.keys())[3]]

('warfarin', 'Vitamin K antagonists')

In [16]:
ingredient_atc4 = pickle.load(open("adr_all_new_up2_26q1_ingredient_fixed.pkl", "rb"))

In [17]:
ingredient_atc4.head()

,caseid,outc_code,pt,yr_qtr,inst
0,31231172,DE; HO,arterial thrombosis; drug interaction; haemorr...,1997Q4,"{""patient"": ""{""age"": ""59.0"", ""age_cod"": ""yr"", ..."
1,32427281,OT,anaemia; dermatitis; ecchymosis,1998Q3,"{""patient"": ""{""age"": ""46.0"", ""age_cod"": ""yr"", ..."
2,37847331,RI,blood pressure decreased; rash erythematous,1998Q4,"{""patient"": ""{""age"": ""72.0"", ""age_cod"": ""yr"", ..."
3,32030921,LT; DE,blister; dermatitis; lip disorder; mucosal ero...,1999Q1,"{""patient"": ""{""age"": ""34.0"", ""age_cod"": ""yr"", ..."
4,32138152,LT; DE,conjunctivitis; mouth ulceration; oral mucosal...,1999Q1,"{""patient"": ""{""age"": ""79.0"", ""age_cod"": ""yr"", ..."


In [18]:
ingredient_atc4.inst[0]


'{"patient": "{"age": "59.0", "age_cod": "yr", "gndr_cod": "f", "wt": "150.0", "wt_cod": "lbs"}", "treatment": "{"drugname": "photofrin", "ingredient": "dihematoporphyrin ether", "atc4": "nan", "route": "intravenous (not otherwise specified)", "dose": "nan nan nan nan"}; {"drugname": "calcium", "ingredient": "calcium carbonate", "atc4": "Calcium compounds", "route": "nan", "dose": "nan nan nan nan"}; {"drugname": "lasix", "ingredient": "furosemide", "atc4": "Sulfonamides, plain", "route": "nan", "dose": "nan nan nan nan"}; {"drugname": "coumadin", "ingredient": "warfarin", "atc4": "Vitamin K antagonists", "route": "nan", "dose": "nan nan nan nan"}; {"drugname": "prevacid", "ingredient": "lansoprazole", "atc4": "Proton pump inhibitors", "route": "nan", "dose": "nan nan nan nan"}; {"drugname": "ferrous sulfate.", "ingredient": "ferrous sulfate", "atc4": "Iron bivalent, oral preparations", "route": "nan", "dose": "nan nan nan nan"}; {"drugname": "dilantin", "ingredient": "phenytoin", "atc

# create BM25 retriever

# map pt to soc

In [18]:
adr_full = pickle.load(open("adr_full_up2_2026_q1.pkl", "rb"))

In [24]:
# prepare pt lst by caseid
pt_lst = [pt.split("; ") for pt in adr_full.pt]

In [25]:
print(len(pt_lst))
pt_lst[0] #3866480 for SOC set

3973842


['arterial thrombosis',
 'drug interaction',
 'haemorrhage',
 'necrosis ischaemic',
 'overdose',
 'porphyria',
 'venous thrombosis']

In [8]:
#pickle.dump(pt_lst, open("pt_full_list.pkl", "wb"))
pt_full = pickle.load(open("pt_full_list.pkl", "rb"))

In [14]:
len(pt_full)

3973842

In [15]:
# check how many pt not in mapping dict
pt_all = [pt for sub_lst in pt_full for pt in sub_lst]

In [16]:
print("Unique pts:", len(set(pt_all)), "Total pts:", len(pt_all) ) #unique pt in full set

Unique pts: 20924 Total pts: 18919327


In [17]:
# load medDRA map
mdhier_df = pickle.load(open("mdhier_pt_soc.pkl", "rb"))

In [18]:
mdhier_df = mdhier_df.apply(lambda x: x.str.lower() if x.dtype == "object" else x)

In [19]:
mdhier_df.head(2)

,pt_code,hlt_code,hlgt_code,soc_code,pt_name,hlt_name,hlgt_name,soc_name,soc_abbrev,pt_soc_code,primary_soc_fg
0,10002043,10002042,10002086,10005329,anaemia folate deficiency,anaemia deficiencies,anaemias nonhaemolytic and marrow depression,blood and lymphatic system disorders,blood,10005329,y
1,10002080,10002042,10002086,10005329,anaemia vitamin b12 deficiency,anaemia deficiencies,anaemias nonhaemolytic and marrow depression,blood and lymphatic system disorders,blood,10005329,y


In [20]:
pt_set = set(pt_all)

In [21]:
len(pt_set)

20924

# check PTs are not in MedDRA 2026 release

In [22]:
pt_out = [i for i in pt_set if i not in list(mdhier_df.pt_name)]

In [23]:
len(pt_out) #1816 for SOC pt set

1820

In [ ]:
#pickle.dump(pt_out, open("pt_out_lst.pkl", "wb"))
#pt_out = pickle.load(open("pt_out_lst.pkl", "rb"))

In [24]:
# recreate mapping_str.pkl
mapping_str = mdhier_df[["pt_name", "soc_name"]].apply(lambda x: "; ".join(x), axis = 1)

In [25]:
mapping_str[:3]

0    anaemia folate deficiency; blood and lymphatic...
1    anaemia vitamin b12 deficiency; blood and lymp...
2    anaemia vitamin b6 deficiency; blood and lymph...
dtype: str

In [77]:
pickle.dump(mapping_str, open("mapping_str.pkl", "wb"))

In [ ]:
# # load pt_out mapped to SOC from ARC
# pt_out_map = pickle.load(open("pt_out_labels.pkl", "rb"))

# map pt_out (most of them actually are llt)

In [28]:
import pandas as pd

In [29]:
# import LLT
llt_file = "MedDRA/MedDRA_Releases/MedDRA_29_0_English/MedAscii/llt.asc"
llt_df = pd.read_csv(llt_file, sep=r"\$", header=None, engine="python")

In [30]:
llt_df.columns = ["llt_code","llt_name","pt_code", "c3", "c4","c5", "c6", "c7",
                  "c8", "llt_currency", "c10", "c11"]
llt_df = llt_df.iloc[:,[0,1,2,9]]

In [31]:
# merge mdhier_df and llt_df by pt_code
llt_soc = pd.merge(llt_df, mdhier_df, on = "pt_code", how = "outer")

In [32]:
len(llt_soc.llt_name.unique())

91082

In [33]:
llt_soc.llt_name = llt_soc.llt_name.str.lower()

In [34]:
in_llt = [i for i in pt_out if i in llt_soc.llt_name.unique()] #1820

In [35]:
# get in_llt df
in_llt_df = llt_soc.loc[llt_soc.llt_name.isin(in_llt), 
                        ["llt_name", "pt_name", "soc_name", "primary_soc_fg"]]

In [36]:
print(in_llt_df.shape)
in_llt_df.head(2)

(2843, 4)


,llt_name,pt_name,soc_name,primary_soc_fg
45,stomach discomfort,abdominal discomfort,gastrointestinal disorders,y
568,abscess of external auditory meatus,abscess of external ear,ear and labyrinth disorders,n


In [37]:
in_llt_df.pt_name = in_llt_df.llt_name

In [38]:
in_llt_df.drop("llt_name", axis = 1, inplace = True)

In [39]:
still_out = [i for i in pt_out if i not in llt_soc.llt_name.unique()]

In [40]:
len(still_out) == len(set(still_out))

True

In [41]:
len(set(still_out)) #same 31 missing PTs 

31

In [42]:
# map 31 new PT to SOC
new_map_pt = {
    "superior vena caval stenosis": "Vascular disorders",
    "her-2 positive gastric cancer": "Neoplasms benign, malignant and unspecified (incl cysts and polyps)",
    "parainfluenzae virus infection": "Infections and infestations",
    "streptococcal identification test positive": "Investigations",
    "pneumonia parainfluenzae viral": "Infections and infestations",
    "methicillin-resistant staphylococcal aureus test positive": "Investigations",
    "staphylococcal identification test positive": "Investigations",
    "eagles syndrome": "Congenital, familial and genetic disorders",
    "superior vena caval occlusion": "Vascular disorders",
    "staphylococcal identification test negative": "Investigations",
    "blastic plasmacytoid dendritric cell neoplasia": "Neoplasms benign, malignant and unspecified (incl cysts and polyps)",
    "epstein barr virus positive mucocutaneous ulcer": "Infections and infestations",
    "streptococcal serology": "Investigations",
    "ano-rectal stenosis": "Gastrointestinal disorders",
    "her-2 positive breast cancer": "Neoplasms benign, malignant and unspecified (incl cysts and polyps)",
    "gastro-jejunostomy": "Surgical and medical procedures",
    "meningeomas surgery": "Surgical and medical procedures",
    "evan's syndrome": "Blood and lymphatic system disorders",
    "capnocytophagia infection": "Infections and infestations",
    "aeromona infection": "Infections and infestations",
    "parainfluenzae viral laryngotracheobronchitis": "Infections and infestations",
    "streptococcal serology positive": "Investigations",
    "methicillin-resistant staphylococcal aureus test negative": "Investigations",
    "gastro-intestinal fistula": "Gastrointestinal disorders",
    "methicillin-resistant staphylococcal aureus test": "Investigations",
    "disbacteriosis": "Gastrointestinal disorders",
    "parainfluenzae viral bronchitis": "Infections and infestations",
    "hypothalamo-pituitary disorders": "Endocrine disorders",
    "frontal sinus operations": "Surgical and medical procedures",
    "parovarian cyst": "Reproductive system and breast disorders",
    "immune-mediated adrenal insuficiency": "Endocrine disorders"
}

In [43]:
len(new_map_pt)

31

In [44]:
new_map_df = pd.DataFrame.from_dict(new_map_pt, orient = "index").reset_index()

In [45]:
new_map_df.columns = ["pt_name", "soc_name"]

In [46]:
new_map_df = new_map_df.apply(lambda x: x.str.lower())

In [47]:
new_map_df["primary_soc_fg"] = "Y"


In [48]:
new_map_df.tail()

,pt_name,soc_name,primary_soc_fg
26,parainfluenzae viral bronchitis,infections and infestations,Y
27,hypothalamo-pituitary disorders,endocrine disorders,Y
28,frontal sinus operations,surgical and medical procedures,Y
29,parovarian cyst,reproductive system and breast disorders,Y
30,immune-mediated adrenal insuficiency,endocrine disorders,Y


In [49]:
pt_soc_all = pd.concat([mdhier_df[["pt_name", "soc_name", "primary_soc_fg"]],
                        new_map_df])

In [50]:
print(pt_soc_all.shape)
pt_soc_all.head(2) #42561

(42561, 3)


,pt_name,soc_name,primary_soc_fg
0,anaemia folate deficiency,blood and lymphatic system disorders,y
1,anaemia vitamin b12 deficiency,blood and lymphatic system disorders,y


In [51]:
pt_soc_all = pd.concat([in_llt_df, pt_soc_all])

In [52]:
print(pt_soc_all.shape)
pt_soc_all.tail(2) #45404

(45404, 3)


,pt_name,soc_name,primary_soc_fg
29,parovarian cyst,reproductive system and breast disorders,Y
30,immune-mediated adrenal insuficiency,endocrine disorders,Y


In [53]:
pt_soc_all.reset_index(drop = True, inplace = True)

In [54]:
pt_soc_all_ = pt_soc_all.apply(lambda x: x.str.lower())

In [2]:
import pickle


In [56]:
pt_soc_all_ = pt_soc_all_[pt_soc_all_.primary_soc_fg == "y"]

In [57]:
pt_soc_all_.shape

(29181, 3)

In [59]:
pickle.dump(pt_soc_all_, open("pt_soc_all_primary.pkl", "wb"))

In [2]:
import pickle

In [7]:
#pickle.dump(pt_soc_all, open("pt_soc_all.pkl", "wb"))
pt_soc_all = pickle.load(open("pt_soc_all.pkl", "rb"))

In [ ]:
pt_soc_all.head()

,pt_name,soc_name,primary_soc_fg
0,stomach discomfort,gastrointestinal disorders,y
1,abscess of external auditory meatus,ear and labyrinth disorders,n
2,abscess of external auditory meatus,infections and infestations,y
3,abscess of external auditory meatus,skin and subcutaneous tissue disorders,n
4,multiple drug overdose accidental,"injury, poisoning and procedural complications",y


In [9]:
pt_soc_all[pt_soc_all.pt_name == "acute myocardial infarction"]

,pt_name,soc_name,primary_soc_fg
4263,acute myocardial infarction,cardiac disorders,y
43618,acute myocardial infarction,vascular disorders,n


In [11]:
sorted(pt_soc_all.soc_name.unique())

['blood and lymphatic system disorders',
 'cardiac disorders',
 'congenital, familial and genetic disorders',
 'ear and labyrinth disorders',
 'endocrine disorders',
 'eye disorders',
 'gastrointestinal disorders',
 'general disorders and administration site conditions',
 'hepatobiliary disorders',
 'immune system disorders',
 'infections and infestations',
 'injury, poisoning and procedural complications',
 'investigations',
 'metabolism and nutrition disorders',
 'musculoskeletal and connective tissue disorders',
 'neoplasms benign, malignant and unspecified (incl cysts and polyps)',
 'nervous system disorders',
 'pregnancy, puerperium and perinatal conditions',
 'product issues',
 'psychiatric disorders',
 'renal and urinary disorders',
 'reproductive system and breast disorders',
 'respiratory, thoracic and mediastinal disorders',
 'skin and subcutaneous tissue disorders',
 'social circumstances',
 'surgical and medical procedures',
 'vascular disorders']

In [6]:
pt_soc_all.soc_name.unique()

array(['gastrointestinal disorders', 'ear and labyrinth disorders',
       'infections and infestations',
       'skin and subcutaneous tissue disorders',
       'injury, poisoning and procedural complications',
       'neoplasms benign, malignant and unspecified (incl cysts and polyps)',
       'congenital, familial and genetic disorders',
       'musculoskeletal and connective tissue disorders',
       'immune system disorders', 'investigations',
       'blood and lymphatic system disorders', 'endocrine disorders',
       'metabolism and nutrition disorders', 'nervous system disorders',
       'psychiatric disorders', 'vascular disorders',
       'respiratory, thoracic and mediastinal disorders', 'eye disorders',
       'surgical and medical procedures',
       'general disorders and administration site conditions',
       'cardiac disorders', 'renal and urinary disorders',
       'hepatobiliary disorders',
       'reproductive system and breast disorders',
       'pregnancy, puerper

In [7]:
pt_soc_all.primary_soc_fg.value_counts()

primary_soc_fg
y    29181
n    16223
Name: count, dtype: int64

In [5]:
# Create multihot SOC labels from nested PT list

# Get unique SOC names sorted for consistent ordering
unique_socs = sorted(pt_soc_all.soc_name.unique())
print(f"Number of unique SOCs: {len(unique_socs)}")
print(unique_socs[:5])

Number of unique SOCs: 27
['blood and lymphatic system disorders', 'cardiac disorders', 'congenital, familial and genetic disorders', 'ear and labyrinth disorders', 'endocrine disorders']


In [6]:
pickle.dump(unique_socs, open("soc_labels_df.pkl", "wb"))    

In [9]:
# Create a mapping from PT name to list of SOCs
pt_to_socs = {}
for _, row in pt_soc_all.iterrows():
    pt_name = row['pt_name']
    soc_name = row['soc_name']
    if pt_name not in pt_to_socs:
        pt_to_socs[pt_name] = []
    if soc_name not in pt_to_socs[pt_name]:
        pt_to_socs[pt_name].append(soc_name)

print(f"Total unique PTs in full set mapping: {len(pt_to_socs)}")

Total unique PTs in full set mapping: 29181


In [10]:
len(pt_to_socs) 

29181

In [11]:
# Create multihot encoding for each sample in pt_lst
# Each row is a binary vector indicating presence of each SOC

def create_multihot_soc_labels(pt_nested_lst, pt_to_socs_map, unique_soc_list):
    """
    Create multihot encoded SOC labels from nested PT lists.
    
    Parameters:
    -----------
    pt_nested_lst : list of lists
        Each element is a list of PTs for that sample
    pt_to_socs_map : dict
        Mapping from PT name to list of SOC names
    unique_soc_list : list
        Sorted list of unique SOC names
        
    Returns:
    --------
    multihot_labels : numpy array (n_samples, n_socs)
        Binary matrix where element [i, j] = 1 if SOC j is present in sample i
    """
    n_samples = len(pt_nested_lst)
    n_socs = len(unique_soc_list)

    print(f"Creating multihot matrix for {n_samples} samples and {n_socs} unique SOCs.")
    
    # Create mapping from SOC name to column index
    soc_to_idx = {soc: idx for idx, soc in enumerate(unique_soc_list)}
    
    # Initialize multihot matrix
    multihot_matrix = np.zeros((n_samples, n_socs), dtype=np.int8)
    
    # Fill in the matrix
    pt_missing = []
    soc_out = []
    for sample_idx, pt_list in enumerate(pt_nested_lst):
        # Collect all unique SOCs for this sample
        socs_in_sample = set()
        for pt in pt_list:
            pt_clean = pt.strip()  # Clean whitespace only
            if pt_clean in pt_to_socs_map:
                # add all SOCs for this PT to the set
                socs_in_sample.update(pt_to_socs_map[pt_clean])
            else:
                pt_missing.append(pt_clean)
        
        # Set binary flags for present SOCs
        for soc in socs_in_sample:
            if soc in soc_to_idx:
                multihot_matrix[sample_idx, soc_to_idx[soc]] = 1
        
        # collect soc by sample_idx
        soc_out.append(list(socs_in_sample))
    
    return soc_out, multihot_matrix, pt_missing


In [12]:
# Load pt full list if not already in memory
if 'pt_full_lst' not in locals():
    pt_full_lst = pickle.load(open("pt_full_list.pkl", "rb"))


In [15]:
len(pt_full_lst)

3973842

In [16]:
# Create multihot labels of SOC for each sample in adr_all (not adr_full)
soc_out, multihot_soc_labels, pt_missing = \
    create_multihot_soc_labels(pt_full_lst, pt_to_socs, unique_socs)

print(f"Multihot SOC labels shape: {multihot_soc_labels.shape}")
print(f"Number of samples: {multihot_soc_labels.shape[0]}")
print(f"Number of SOC dimensions: {multihot_soc_labels.shape[1]}")
print(f"\nSOC dimension names (in order):")
for i, soc in enumerate(unique_socs):
    count = multihot_soc_labels[:, i].sum()
    print(f"  [{i:2d}] {soc}: {count} samples")

Creating multihot matrix for 3973842 samples and 27 unique SOCs.
Multihot SOC labels shape: (3973842, 27)
Number of samples: 3973842
Number of SOC dimensions: 27

SOC dimension names (in order):
  [ 0] blood and lymphatic system disorders: 365187 samples
  [ 1] cardiac disorders: 804853 samples
  [ 2] congenital, familial and genetic disorders: 31664 samples
  [ 3] ear and labyrinth disorders: 91542 samples
  [ 4] endocrine disorders: 184468 samples
  [ 5] eye disorders: 241185 samples
  [ 6] gastrointestinal disorders: 1094507 samples
  [ 7] general disorders and administration site conditions: 1677534 samples
  [ 8] hepatobiliary disorders: 207089 samples
  [ 9] immune system disorders: 534138 samples
  [10] infections and infestations: 726548 samples
  [11] injury, poisoning and procedural complications: 1201681 samples
  [12] investigations: 793068 samples
  [13] metabolism and nutrition disorders: 550418 samples
  [14] musculoskeletal and connective tissue disorders: 742642 sample

In [17]:
print(len(soc_out))
soc_out[0]

3973842


['congenital, familial and genetic disorders',
 'injury, poisoning and procedural complications',
 'vascular disorders',
 'metabolism and nutrition disorders',
 'general disorders and administration site conditions']

In [18]:
len(set(pt_missing)) #all pt are mapped to SOC, no pt_out

0

In [19]:
# Create a DataFrame with SOC labels for easier handling
soc_labels_df = pd.DataFrame(multihot_soc_labels, columns=unique_socs)
print(f"\nDataFrame shape: {soc_labels_df.shape}")
print(f"Labels per sample (mean): {multihot_soc_labels.sum(axis=1).mean():.2f}")
print(f"Labels per sample (min/max): {multihot_soc_labels.sum(axis=1).min()}/{multihot_soc_labels.sum(axis=1).max()}")


DataFrame shape: (3973842, 27)
Labels per sample (mean): 3.90
Labels per sample (min/max): 1/27


In [20]:
# Save the multihot labels
np.save("multihot_soc_labels.npy", multihot_soc_labels)
pickle.dump(multihot_soc_labels, open("multihot_soc_labels.pkl", "wb"))
pickle.dump(soc_labels_df, open("soc_labels_df.pkl", "wb"))
print("\nMultihot labels saved to:")
print("  - multihot_soc_labels.npy (numpy array)")
print("  - multihot_soc_labels.pkl (numpy array)")
print("  - soc_labels_df.pkl (DataFrame with SOC names as columns)")


Multihot labels saved to:
  - multihot_soc_labels.npy (numpy array)
  - multihot_soc_labels.pkl (numpy array)
  - soc_labels_df.pkl (DataFrame with SOC names as columns)


In [22]:
multihot_soc_labels.shape

(3973842, 27)

In [28]:
soc_lbl_zip = dict(zip(adr_full.caseid, multihot_soc_labels))

In [34]:
# # zip soc label adn caseid
soc_pt_zip = dict(zip(adr_full.caseid, soc_out))

In [32]:
soc_out[0]

['congenital, familial and genetic disorders',
 'injury, poisoning and procedural complications',
 'vascular disorders',
 'metabolism and nutrition disorders',
 'general disorders and administration site conditions']

In [31]:
soc_lbl_zip[adr_full.caseid[0]]

array([0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 1], dtype=int8)

In [39]:
adr_all.shape

(3866480, 5)

In [40]:
adr_all["soc"] = [soc_lbl_zip[i] for i in adr_all.caseid]

In [44]:
adr_all["soc_lbl"] = [soc_pt_zip[i] for i in adr_all.caseid]

In [ ]:
adr_all.iloc[3866314]

caseid                                               262533943
outc_code                                               DE; OT
pt           ileus; dehydration; faecaloma; dizziness; canc...
yr_qtr                                                  2026Q1
inst         {"patient": "{"age": "64.0", "age_cod": "yr", ...
soc          [0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, ...
soc_lbl      [neoplasms benign, malignant and unspecified (...
Name: 3866314, dtype: object

In [47]:
adr_all.soc_lbl[3866314]

['neoplasms benign, malignant and unspecified (incl cysts and polyps)',
 'infections and infestations',
 'gastrointestinal disorders',
 'nervous system disorders',
 'vascular disorders',
 'cardiac disorders',
 'metabolism and nutrition disorders',
 'general disorders and administration site conditions']

In [51]:
for i in adr_all.pt[3866314].split("; "):
    print(i, "->", pt_soc_all.loc[pt_soc_all.pt_name == i, "soc_name"].tolist())

ileus -> ['gastrointestinal disorders']
dehydration -> ['metabolism and nutrition disorders']
faecaloma -> ['gastrointestinal disorders']
dizziness -> ['cardiac disorders', 'nervous system disorders', 'vascular disorders']
cancer pain -> ['neoplasms benign, malignant and unspecified (incl cysts and polyps)']
decreased appetite -> ['general disorders and administration site conditions', 'metabolism and nutrition disorders']
bacterial translocation -> ['infections and infestations']
headache -> ['nervous system disorders']
diarrhoea -> ['gastrointestinal disorders']
circulatory collapse -> ['vascular disorders']
sepsis -> ['infections and infestations']


# soc zip for GraphDB input

In [11]:
#pickle.dump(soc_pt_zip, open("caseid_soc_pt_zip.pkl", "wb"))
soc_pt_zip = pickle.load(open("caseid_soc_pt_zip.pkl", "rb"))

In [15]:
len(soc_pt_zip)

3973842

In [ ]:
list(soc_pt_zip.keys())[0]

31231172

In [20]:
list(soc_pt_zip.values())[0]

['congenital, familial and genetic disorders',
 'injury, poisoning and procedural complications',
 'vascular disorders',
 'metabolism and nutrition disorders',
 'general disorders and administration site conditions']

In [8]:
#pickle.dump(soc_lbl_zip, open("caseid_soc_lbl_zip.pkl", "wb"))
soc_lbl_zip = pickle.load(open("caseid_soc_lbl_zip.pkl", "rb"))

In [10]:
type(soc_lbl_zip)


dict

In [13]:
import gc

In [14]:
#del soc_lbl_zip
gc.collect()

620

In [60]:
print(soc_labels_df.shape)
soc_labels_df.head() #3973842

(3973842, 27)


,blood and lymphatic system disorders,cardiac disorders,"congenital, familial and genetic disorders",ear and labyrinth disorders,endocrine disorders,eye disorders,gastrointestinal disorders,general disorders and administration site conditions,hepatobiliary disorders,immune system disorders,...,"pregnancy, puerperium and perinatal conditions",product issues,psychiatric disorders,renal and urinary disorders,reproductive system and breast disorders,"respiratory, thoracic and mediastinal disorders",skin and subcutaneous tissue disorders,social circumstances,surgical and medical procedures,vascular disorders
0,0,0,1,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,1
1,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,1
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
3,0,0,0,0,0,0,1,1,0,1,...,0,0,0,0,0,0,1,0,0,1
4,0,0,0,0,0,1,1,0,0,1,...,0,0,0,0,0,0,1,0,0,1


In [22]:
soc_labels_df.iloc[0, [i for i, val in enumerate(soc_labels_df.iloc[0]) if val == 1]]

congenital, familial and genetic disorders              1
general disorders and administration site conditions    1
injury, poisoning and procedural complications          1
vascular disorders                                      1
Name: 0, dtype: int8

In [26]:
for i in pt_lst[0]:
    print(f"{i}: {pt_to_socs[i]}")

arterial thrombosis: ['vascular disorders']
drug interaction: ['general disorders and administration site conditions']
haemorrhage: ['vascular disorders']
necrosis ischaemic: ['vascular disorders']
overdose: ['injury, poisoning and procedural complications']
porphyria: ['congenital, familial and genetic disorders']
venous thrombosis: ['vascular disorders']


# prepare SOC out for SOC set   

In [ ]:
# training data limited to 2024
adr_trn = adr_all[adr_all.yr_qtr <= "2024Q4"]
adr_tst = adr_all[adr_all.yr_qtr > "2024Q4"]

In [15]:
pickle.dump((adr_trn, adr_tst), open("./hybrid_rag/adr_trn_tst_627.pkl", "wb"))

In [27]:
(adr_trn, adr_tst) = pickle.load(open("./hybrid_rag/adr_trn_tst_627.pkl", "rb"))

In [28]:
print(adr_tst.shape)
adr_tst.head(2)

(272548, 5)


,caseid,outc_code,pt,yr_qtr,inst
3593932,247958291,OT,weight decreased; prostate cancer; thrombosis;...,2025Q1,"{""patient"": ""{""age"": ""76.0"", ""age_cod"": ""yr"", ..."
3593933,247958231,OT,chills; tachycardia,2025Q1,"{""patient"": ""{""age"": ""50.0"", ""age_cod"": ""yr"", ..."


In [20]:
ls -al ./hybrid_rag/adr_trn_tst_627.pkl 

-rw-rw-r-- 1 dada dada 5038297116 Jun 27 08:30 ./hybrid_rag/adr_trn_tst_627.pkl


In [29]:
inst_lst = adr_trn.inst.tolist()
len(inst_lst)

3593932

In [82]:
adr_trn.inst[0]

'{"patient": "{"age": "59.0", "age_cod": "yr", "gndr_cod": "f", "wt": "150.0", "wt_cod": "lbs"}", "treatment": "{"drugname": "photofrin", "ingredient": "dihematoporphyrin ether", "atc4": "nan", "route": "intravenous (not otherwise specified)", "dose": "nan nan nan nan"}; {"drugname": "calcium", "ingredient": "calcium", "atc4": "calcium", "route": "nan", "dose": "nan nan nan nan"}; {"drugname": "lasix", "ingredient": "furosemide", "atc4": "sulfonamides, plain", "route": "nan", "dose": "nan nan nan nan"}; {"drugname": "coumadin", "ingredient": "warfarin", "atc4": "vitamin k antagonists", "route": "nan", "dose": "nan nan nan nan"}; {"drugname": "prevacid", "ingredient": "lansoprazole", "atc4": "proton pump inhibitors", "route": "nan", "dose": "nan nan nan nan"}; {"drugname": "ferrous sulfate.", "ingredient": "ferrous sulfate.", "atc4": "ferrous sulfate.", "route": "nan", "dose": "nan nan nan nan"}; {"drugname": "dilantin", "ingredient": "phenytoin", "atc4": "hydantoin derivatives", "route

In [16]:
oot_lst = adr_tst.inst.tolist() 

In [23]:
pickle.dump(inst_lst, open("inst_trn.pkl", "wb"))
pickle.dump(oot_lst, open("oot_inst_lst.pkl", "wb"))

In [84]:
ls -al inst_trn.pkl

/home/dada/anaconda3/lib/python3.12/pty.py:95: RuntimeWarning: lancedb fork support is experimental: the internal async runtime has been reset in the forked child, but a small chance of deadlock remains if other state was mid-operation at fork time. The 'forkserver' or 'spawn' multiprocessing start method is likely a safer alternative.
  pid, fd = os.forkpty()


-rw-rw-r-- 1 dada dada 4101781376 Jun 27 14:09 inst_trn.pkl


# Create BM25 retriever on adr_trn.inst

In [2]:
import pickle

In [43]:
#inst_lst = pickle.load(open("inst_trn.pkl", "rb"))
oot_lst = pickle.load(open("oot_inst_lst.pkl", "rb"))

In [36]:
len(oot_lst)

272548

In [37]:
# %%time

# from multiprocessing import Pool

# with Pool(12) as p:
#     tokenized_docs = list(
#         p.imap(
#             bm25_faers_tokenizer,
#             oot_lst,
#             chunksize=5000
#         )
#     )

In [38]:
# pickle.dump(tokenized_docs, open("tokenized_tst_inst.pkl", "wb"))
# tokenized_docs = pickle.load(open("tokenized_tst_inst.pkl", "rb"))

# Index tokenized inst

In [2]:
import bm25s

In [ ]:
retriever = bm25s.BM25()

In [46]:
trn_lst = pickle.load(open("inst_trn.pkl", "rb"))

In [47]:
len(trn_lst)

3593932

In [48]:
# tokenize trn_lst
trn_tokenized = bm25s.tokenize(trn_lst)

Split strings:   0%|          | 0/3593932 [00:00<?, ?it/s]

In [49]:
# index the tokenized training instances
retriever.index(trn_tokenized)

BM25S Count Tokens:   0%|          | 0/3593932 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/3593932 [00:00<?, ?it/s]

In [50]:
# save bm25 retriever to disk for later use
retriever.save("bm25_trn_retreiver")

In [41]:
retriever = bm25s.BM25.load("bm25_trn_retreiver", mmap=True)

In [ ]:
#oot_tokenized_docs = pickle.load(open("tokenized_tst_inst.pkl", "rb"))

In [35]:
oot_tokenized = bm25s.tokenize(oot_lst)

Split strings:   0%|          | 0/272548 [00:00<?, ?it/s]

In [39]:
#pickle.dump((trn_tokenized, oot_tokenized), open("trn_oot_tokenized.pkl", "wb"))
trn_tokenized, oot_tokenized= pickle.load(open("trn_oot_tokenized.pkl", "rb"))

In [4]:
del trn_tokenized
gc.collect()

20

In [35]:
from multiprocessing import Pool, cpu_count
from tqdm.notebook import tqdm, trange
import pickle, time
import bm25s
from bm25s.tokenization import Tokenized
from tqdm.contrib.concurrent import process_map #see each worker progress

trn_tokenized, oot_tokenized= pickle.load(open("trn_oot_tokenized.pkl", "rb"))

retriever = bm25s.BM25()
#retriever.index(trn_tokenized)
retriever = bm25s.BM25.load("bm25_trn_retreiver")

# chunks = [
#     oot_tokenized[0][i:i+8192]
#     for i in range(0, len(oot_tokenized[0]), 8192)
# ]


In [76]:
def retrieve_chunk(tokenized_queries):
    global retriever
    return retriever.retrieve(tokenized_queries, k=5)

chunks = [
    Tokenized(
        ids=oot_tokenized.ids[i:i+8192],
        vocab=oot_tokenized.vocab
    )
    for i in range(0, len(oot_tokenized.ids), 8192)
]


In [ ]:
%%time 

results = process_map(retrieve_chunk, chunks, max_workers=cpu_count())

  0%|          | 0/34 [00:00<?, ?it/s]

# load BM25 matchs from ARC

In [34]:
bm25_match = pickle.load(open("bm25_oot_pool.pkl", "rb"))

In [35]:
bm25_match[32][0]

array([[1236479,  957609, 1080582, 1264530, 3134466],
       [3366396, 3582490, 3571800, 3498042, 3283663],
       [3071328, 3308372, 1706455, 1707493, 3318097],
       ...,
       [1855887, 3484981, 3526449, 3078104, 2404171],
       [3507572, 1445875, 3493890,  906269, 3575926],
       [3426453, 3428401, 3514670, 3462088, 3480678]], shape=(8192, 5))

In [36]:
bm25_match[32][1].shape

(8192, 5)

In [37]:
# merge 34 chunks
bm25_ids = np.concatenate([chunk[0].astype(np.int32) for chunk in bm25_match])
bm25_scores = np.concatenate([chunk[1].astype(np.float64) for chunk in bm25_match])

In [28]:
bm25_scores.shape

(272548, 5)

In [29]:
bm25_ids[0]

array([3227685, 2201309, 3274897, 3287041, 2859269], dtype=int32)

In [30]:
bm25_scores[0]

array([100.612854  , 100.36943817,  98.56613159,  95.6333847 ,
        95.56513977])

In [38]:
bm25_scores_flat = np.array([s for row in bm25_scores for s in row], dtype=np.float32)
has_nan = np.isnan(bm25_scores_flat).any()
print(f"BM25 scores contain NaN: {has_nan}")
if has_nan:
    print("NaN count:", int(np.isnan(bm25_scores_flat).sum()))
    print("NaN positions:", np.argwhere(np.isnan(bm25_scores_flat)).tolist())

BM25 scores contain NaN: False


In [39]:
pickle.dump((bm25_ids, bm25_scores), open("bm25_soc_oot_ids_scores.pkl", "wb"))

In [44]:
# test a few matches
for i in [0, 100000, 200000]:
    retrieved_indices = bm25_ids[i]
    retrieved_scores = bm25_scores[i]

    # Print the results for the current query index
    ids, scores = retriever.retrieve(bm25s.tokenize(oot_lst[i]), k= 5)   
    
    print(f"\nQuery index: {i}")
    print(f"Retrieved indices: {retrieved_indices}")
    print(f"Retrieved scores: {retrieved_scores}")  

    print(f"Test indices: {ids}")
    print(f"Test scores: {scores}")        

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


Query index: 0
Retrieved indices: [3227685 2201309 3274897 3287041 2859269]
Retrieved scores: [100.612854   100.36943817  98.56613159  95.6333847   95.56513977]
Test indices: [[3227685 2201309 3274897 3287041 2859269]]
Test scores: [[100.612854 100.36944   98.56613   95.633385  95.56514 ]]


Split strings:   0%|          | 0/1 [00:02<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


Query index: 100000
Retrieved indices: [3470710 3546574 3527806 3569299 3426659]
Retrieved scores: [24.96818542 24.58037567 24.58037567 24.58037567 24.54963875]
Test indices: [[3470710 3527806 3569299 3546574 3426659]]
Test scores: [[24.968185 24.580376 24.580376 24.580376 24.549639]]


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


Query index: 200000
Retrieved indices: [3314848 3306990 3532487 3343959 3308505]
Retrieved scores: [202.81256104 201.59254456 201.09721375 200.05023193 198.68429565]
Test indices: [[3314848 3306990 3532487 3343959 3308505]]
Test scores: [[202.81256 201.59254 201.09721 200.05023 198.6843 ]]


# prepare 3:3 triplet data for RAG training - group-based samples:
* anchor
* positives: cases sharing most SOC labels
* hard negatives: cases with similar context but different outcomes

In [ ]:
trn_idx

In [16]:
trn_idx[-1]

3593931

In [31]:
adr_all = pickle.load(open("adr_all_new_up2_26q1.pkl", "rb")) 

In [36]:
adr_all.shape

(3866480, 5)

In [8]:
adr_all.head()

,caseid,outc_code,pt,yr_qtr,inst
0,31231172,DE; HO,arterial thrombosis; drug interaction; haemorr...,1997Q4,"{""patient"": ""{""age"": ""59.0"", ""age_cod"": ""yr"", ..."
1,32427281,OT,anaemia; dermatitis; ecchymosis,1998Q3,"{""patient"": ""{""age"": ""46.0"", ""age_cod"": ""yr"", ..."
2,37847331,RI,blood pressure decreased; rash erythematous,1998Q4,"{""patient"": ""{""age"": ""72.0"", ""age_cod"": ""yr"", ..."
3,32030921,LT; DE,blister; dermatitis; lip disorder; mucosal ero...,1999Q1,"{""patient"": ""{""age"": ""34.0"", ""age_cod"": ""yr"", ..."
4,32138152,LT; DE,conjunctivitis; mouth ulceration; oral mucosal...,1999Q1,"{""patient"": ""{""age"": ""79.0"", ""age_cod"": ""yr"", ..."


In [10]:
len(soc_zip)

3973842

In [46]:
multihot_soc_labels = pickle.load(open("multihot_soc_labels.pkl", "rb"))

In [80]:
row_0 = multihot_soc_labels.sum(axis =1)

In [81]:
[i==0 for i in row_0].count(True)

0

In [82]:
adr_full.iloc[zero_rows[0]]

caseid                                                 4262993
outc_code                                                   HO
pt                convulsion; pharmaceutical product complaint
yr_qtr                                                  2004Q1
inst         {"patient": "{"age": "64.0", "age_cod": "yr", ...
Name: 173, dtype: object

In [83]:
multihot_soc_labels[zero_rows[0]]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0,
       0, 0, 0, 0, 0], dtype=int8)

In [35]:
soc_lbl_zip[31231172]

array([0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 1], dtype=int8)

In [49]:
# create adr_trn SOC labels
adr_all["soc"] = [soc_lbl_zip[idx] for idx in list(adr_all.caseid)]

In [37]:
adr_all.shape

(3866480, 6)

In [2]:
import pickle

In [3]:
#pickle.dump(adr_all, open("adr_prim_lbl_all_soc_lbl.pkl", "wb"))
adr_all = pickle.load(open("adr_lbl_all_soc_lbl.pkl", "rb"))

In [6]:
adr_all.soc[3760056]

array([0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 1], dtype=int8)

In [30]:
print(adr_all.shape)
adr_all[(adr_all.outc_code.str.contains("DE")) & (adr_all.outc_code is not None)].tail(20)

(3866480, 6)


,caseid,outc_code,pt,yr_qtr,inst,soc
3866160,265590151,DE; OT,cardiac arrest; upper gastrointestinal haemorr...,2026Q1,"{""patient"": ""{""age"": ""60.0"", ""age_cod"": ""yr"", ...","[0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3866201,265303282,DE,acute myocardial infarction,2026Q1,"{""patient"": ""{""age"": ""84.0"", ""age_cod"": ""yr"", ...","[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3866224,265579931,DE,death,2026Q1,"{""patient"": ""{""age"": ""98.0"", ""age_cod"": ""yr"", ...","[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, ..."
3866227,265249692,DE,death,2026Q1,"{""patient"": ""{""age"": ""58.0"", ""age_cod"": ""yr"", ...","[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, ..."
3866253,265600681,DS; HO; LT; OT; DE,swollen joint count increased; therapeutic res...,2026Q1,"{""patient"": ""{""age"": ""43.0"", ""age_cod"": ""yr"", ...","[0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, ..."
3866284,265270902,OT; LT; HO; DE,septic shock; generalised tonic-clonic seizure...,2026Q1,"{""patient"": ""{""age"": ""6.0"", ""age_cod"": ""mon"", ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, ..."
3866288,265267722,OT; DE; HO,pyrexia; bacterial sepsis; therapeutic respons...,2026Q1,"{""patient"": ""{""age"": ""71.0"", ""age_cod"": ""yr"", ...","[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, ..."
3866302,265148992,CA; DE; DS; HO; LT; OT,folliculitis; fall; musculoskeletal stiffness;...,2026Q1,"{""patient"": ""{""age"": ""40.0"", ""age_cod"": ""yr"", ...","[0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3866314,262533943,DE; OT,ileus; dehydration; faecaloma; dizziness; canc...,2026Q1,"{""patient"": ""{""age"": ""64.0"", ""age_cod"": ""yr"", ...","[0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, ..."
3866319,2324136110,OT; DE,pulmonary oedema; dermatitis bullous; iron def...,2026Q1,"{""patient"": ""{""age"": ""75.0"", ""age_cod"": ""yr"", ...","[1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, ..."


In [62]:
display(Markdown(adr_all.loc[adr_all.caseid == 262533943, "pt"].values[0].replace("; ", "\n")))

ileus
dehydration
faecaloma
dizziness
cancer pain
decreased appetite
bacterial translocation
headache
diarrhoea
circulatory collapse
sepsis

In [ ]:
adr_all.loc[adr_all.caseid == 262533943, "soc"].values[0] #8

array([0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 1], dtype=int8)

In [17]:
from IPython.display import display, Markdown

In [ ]:
display(Markdown(f"**Instance 3866314:** {adr_all.loc[3866314, 'soc']}"))

**Instance 3866314:** [0 1 0 0 0 0 1 1 0 0 1 0 0 1 0 1 1 0 0 0 0 0 0 0 0 0 1]

In [ ]:
display(Markdown(f"**Instance 3866314:** {adr_all.loc[3866314, 'pt']}"))

**Instance 3866314:** ileus; dehydration; faecaloma; dizziness; cancer pain; decreased appetite; bacterial translocation; headache; diarrhoea; circulatory collapse; sepsis

In [35]:
display(Markdown(f"**Instance 3866314:** {adr_all.loc[3866314, 'inst']}"))

**Instance 3866314:** {"patient": "{"age": "64.0", "age_cod": "yr", "gndr_cod": "m", "wt": "75.0", "wt_cod": "kg"}", "treatment": "{"drugname": "magnesium", "ingredient": "magnesium", "atc4": "magnesium", "route": "nan", "dose": "nan nan nan nan"}; {"drugname": "diphenhydramine", "ingredient": "diphenhydramine", "atc4": "aminoalkyl ethers, antihistamines for topical use", "route": "nan", "dose": "nan nan cream nan"}; {"drugname": "lenvima", "ingredient": "lenvatinib", "atc4": "other protein kinase inhibitors", "route": "oral use", "dose": "20.0 mg capsule qd"}; {"drugname": "lenvima", "ingredient": "lenvatinib", "atc4": "other protein kinase inhibitors", "route": "oral use", "dose": "10.0 mg capsule qd"}; {"drugname": "glycosaminoglycans", "ingredient": "glycosaminoglycans", "atc4": "glycosaminoglycans", "route": "nan", "dose": "nan nan cream nan"}; {"drugname": "keytruda", "ingredient": "pembrolizumab", "atc4": "pd-1/pd-l1 (programmed cell death protein 1/death ligand 1) inhibitors", "route": "intravenous drip", "dose": "200.0 mg injection q3w"}; {"drugname": "lenvima", "ingredient": "lenvatinib", "atc4": "other protein kinase inhibitors", "route": "oral use", "dose": "4.0 mg capsule qod"}; {"drugname": "lenvima", "ingredient": "lenvatinib", "atc4": "other protein kinase inhibitors", "route": "oral use", "dose": "4.0 mg capsule qd"}", "indi_pt": "renal cell carcinoma stage iv; metastatic renal cell carcinoma; metastatic renal cell carcinoma; renal cell carcinoma stage iv", "yr_qtr": "2026q1"}

In [41]:
adr_trn_soc = adr_all.loc[trn_idx, ["caseid","yr_qtr", "inst", "soc"]]
adr_tst_soc = adr_all.loc[tst_idx, ["caseid","yr_qtr", "inst", "soc"]]

In [42]:
# prep primary only SOC
pickle.dump(adr_trn_soc, open("adr_trn_prim_soc.pkl", "wb"))
pickle.dump(adr_tst_soc, open("adr_tst_prim_soc.pkl", "wb"))

# create 1:1:3 training data for ft bge_m3

In [ ]:
random.seed(1234)

# keep only valid inst-pt pairs
triplet_source = adr_trn[['inst', 'pt']].dropna().reset_index(drop=True)
triplet_source = triplet_source[
    (triplet_source['inst'].astype(str).str.strip() != "") &
    (triplet_source['pt'].astype(str).str.strip() != "")
].reset_index(drop=True)

# Drop exact duplicate (inst, pt) pairs to avoid repeated training examples.
triplet_source = triplet_source.drop_duplicates().reset_index(drop=True)

if triplet_source.empty:
    raise ValueError("No valid inst-pt pairs are available to build triplets.")

n = len(triplet_source)
num_pairs = 1_000_000
negatives_per_positive = 3
triplets = []

for _ in tqdm(range(num_pairs), desc="Create triplets"):
    idx = random.randrange(n)
    anchor = triplet_source.at[idx, 'inst']
    positive = triplet_source.at[idx, 'pt']

    for _ in range(negatives_per_positive):
        neg_idx = random.randrange(n)
        while triplet_source.at[neg_idx, 'pt'] == positive:
            neg_idx = random.randrange(n)
        negative = triplet_source.at[neg_idx, 'pt']
        triplets.append((anchor, positive, negative))

triplet_df = pd.DataFrame(triplets, columns=['anchor', 'positive', 'negative'])

# Test ft embedding model 

In [ ]:
# #!zip faers_trn_bm25.zip -r faers_trn_bm25
# import shutil
# shutil.make_archive('faers_trn_bm25', 'zip', './faers_trn_bm25')

In [52]:
from sentence_transformers import SentenceTransformer
import torch

In [54]:
model = SentenceTransformer("/home/dada/Barn/GQ/ADR/hybrid_rag/bge_m3_triplet", 
                            model_kwargs={"dtype": torch.float16})

In [56]:
tt='{"patient": "{"age": "64.0", "age_cod": "yr", "gndr_cod": "m", "wt": "75.0", "wt_cod": "kg"}", "treatment": "{"drugname": "magnesium", "ingredient": "magnesium", "atc4": "magnesium", "route": "nan", "dose": "nan nan nan nan"}; {"drugname": "diphenhydramine", "ingredient": "diphenhydramine", "atc4": "aminoalkyl ethers, antihistamines for topical use", "route": "nan", "dose": "nan nan cream nan"}; {"drugname": "lenvima", "ingredient": "lenvatinib", "atc4": "other protein kinase inhibitors", "route": "oral use", "dose": "20.0 mg capsule qd"}; {"drugname": "lenvima", "ingredient": "lenvatinib", "atc4": "other protein kinase inhibitors", "route": "oral use", "dose": "10.0 mg capsule qd"}; {"drugname": "glycosaminoglycans", "ingredient": "glycosaminoglycans", "atc4": "glycosaminoglycans", "route": "nan", "dose": "nan nan cream nan"}; {"drugname": "keytruda", "ingredient": "pembrolizumab", "atc4": "pd-1/pd-l1 (programmed cell death protein 1/death ligand 1) inhibitors", "route": "intravenous drip", "dose": "200.0 mg injection q3w"}; {"drugname": "lenvima", "ingredient": "lenvatinib", "atc4": "other protein kinase inhibitors", "route": "oral use", "dose": "4.0 mg capsule qod"}; {"drugname": "lenvima", "ingredient": "lenvatinib", "atc4": "other protein kinase inhibitors", "route": "oral use", "dose": "4.0 mg capsule qd"}", "indi_pt": "renal cell carcinoma stage iv; metastatic renal cell carcinoma; metastatic renal cell carcinoma; renal cell carcinoma stage iv", "yr_qtr": "2026q1"}'

In [59]:
query_vec_ = model.encode(
    [adr_all.inst[3866314]],
    normalize_embeddings=True,
    convert_to_numpy=True
)[0]

In [60]:
query_vec_

array([ 0.01515 , -0.006317,  0.03087 , ...,  0.02869 ,  0.006718,
       -0.00535 ], shape=(1024,), dtype=float16)

In [58]:
query_vec

array([ 0.01515 , -0.006317,  0.03087 , ...,  0.02869 ,  0.006718,
       -0.00535 ], shape=(1024,), dtype=float16)

In [ ]:
# # training data limited to 2024
# adr_trn = adr_all[adr_all.yr_qtr <= "2024Q4"]
# adr_tst = adr_all[adr_all.yr_qtr > "2024Q4"]

In [47]:
random.seed(1234)

# keep only valid inst-pt pairs
triplet_source = adr_trn[['inst', 'pt']].dropna().reset_index(drop=True)
triplet_source = triplet_source[
    (triplet_source['inst'].astype(str).str.strip() != "") &
    (triplet_source['pt'].astype(str).str.strip() != "")
].reset_index(drop=True)

# Drop exact duplicate (inst, pt) pairs to avoid repeated training examples.
triplet_source = triplet_source.drop_duplicates().reset_index(drop=True)

if triplet_source.empty:
    raise ValueError("No valid inst-pt pairs are available to build triplets.")

n = len(triplet_source)
num_pairs = 1_000_000
negatives_per_positive = 3
triplets = []

for _ in tqdm(range(num_pairs), desc="Create triplets"):
    idx = random.randrange(n)
    anchor = triplet_source.at[idx, 'inst']
    positive = triplet_source.at[idx, 'pt']

    for _ in range(negatives_per_positive):
        neg_idx = random.randrange(n)
        while triplet_source.at[neg_idx, 'pt'] == positive:
            neg_idx = random.randrange(n)
        negative = triplet_source.at[neg_idx, 'pt']
        triplets.append((anchor, positive, negative))

triplet_df = pd.DataFrame(triplets, columns=['anchor', 'positive', 'negative'])
# triplet_df.to_pickle("adr_bge_m3_triplets_1M.pkl")


Create triplets:   0%|          | 0/1000000 [00:00<?, ?it/s]

In [48]:
print(triplet_df.shape)
triplet_df.head()

(3000000, 3)


,anchor,positive,negative
0,"{""patient"": ""{""age"": ""67.0"", ""age_cod"": ""yr"", ...",rectal abscess,drug effect decreased
1,"{""patient"": ""{""age"": ""67.0"", ""age_cod"": ""yr"", ...",rectal abscess,blood pressure decreased; respiratory depression
2,"{""patient"": ""{""age"": ""67.0"", ""age_cod"": ""yr"", ...",rectal abscess,blood glucose decreased; blood glucose increas...
3,"{""patient"": ""{""age"": ""13.0"", ""age_cod"": nan, ""...",crohn's disease,acute kidney injury; febrile neutropenia
4,"{""patient"": ""{""age"": ""13.0"", ""age_cod"": nan, ""...",crohn's disease,drug interaction; product use in unapproved in...


In [49]:
pickle.dump(triplet_df, open("adr_bge_m3_triplets_1M.pkl", "wb"))

In [ ]:
s_len = [len(s.split(" ")) for s in triplet_df.anchor]

In [ ]:
triplet_df.head()

,anchor,positive,negative
0,"{""patient"": ""{""age"": ""65.0"", ""age_cod"": ""yr"", ...",blood creatinine increased; hydronephrosis; ki...,device use error
1,"{""patient"": ""{""age"": ""63.0"", ""age_cod"": ""yr"", ...",alanine aminotransferase increased; aspartate ...,drug level increased; haemoglobin abnormal
2,"{""patient"": ""{""age"": ""53.0"", ""age_cod"": ""yr"", ...",arthralgia; injection site bruising; pain,abdominal pain upper; aphagia; back pain; ches...
3,"{""patient"": ""{""age"": ""15.0"", ""age_cod"": ""yr"", ...",myelosuppression,condition aggravated; decreased appetite; dysp...
4,"{""patient"": ""{""age"": ""36.0"", ""age_cod"": ""yr"", ...",erectile dysfunction; priapism,coma; dyspnoea; malignant neoplasm progression


In [ ]:
sum([s > 512 for s in s_len])

17932

In [ ]:
s_len.index(max(s_len))

48066

In [ ]:
ss = triplet_df.anchor[48066]
ss[:100]

'{"patient": "{"age": "80.0", "age_cod": "yr", "gndr_cod": "m", "wt": "69.0", "wt_cod": "kg"}", "trea'

In [ ]:
len(ss.split(" "))

25321

# Compare 1:3 embedding model performance 

In [8]:
def _rank_observed_in_candidates(
    model: SentenceTransformer,
    query_inst: str,
    observed_outcome: str,
    adr_random_outcomes: list[str],
    k: int = 5,
) -> dict:
    """
    Returns rank (1-based) of observed among candidates, plus top-k list.

    Scoring: cosine(sim(emb(query_inst), emb(candidate_outcome_pt))).
    """
    # Candidates = retrieved outcomes + observed outcome (ensure included)
    fused_pts = adr_random_outcomes + [observed_outcome] #5+1

    # Keep first occurrence order but unique (important if observed already present)
    seen = set()
    uniq_cand_pts = []
    for c in fused_pts:
        if c not in seen:
            uniq_cand_pts.append(c)
            seen.add(c)

    q_emb = model.encode([query_inst], normalize_embeddings=True)
    d_embs = model.encode(uniq_cand_pts, normalize_embeddings=True)

    # get cosine similarity and rank
    sims = (q_emb @ d_embs.T).ravel().astype(float)
    order = np.argsort(-sims)
    ranked = [(uniq_cand_pts[i], sims[i]) for i in order]

    # rank of observed is 1-based index of observed in ranked list
    obs_idx = next(i for i, (txt, _) in enumerate(ranked) if txt == fused_pts[-1])
    rank = obs_idx + 1

    return {
        "rank": rank,
        "hit@1": 1 if rank <= 1 else 0,
        "hit@3": 1 if rank <= 3 else 0,
        "hit@5": 1 if rank <= 5 else 0,
        "mrr@5": (1.0 / rank) if rank <= 5 else 0.0,
        # single-relevant-item nDCG@5
        "ndcg@5": float((1.0 / np.log2(rank + 1)) if rank <= 5 else 0.0),
        "topk": ranked[:k],
    }


def evaluate_embed_models(
    model_names: list[str],
    adr_tst: pd.DataFrame,
    adr_trn: pd.DataFrame,         
    sample_idx: list[int],
    device: str = "cuda",
    random_seed = 1234
) -> pd.DataFrame:
    """
    For each model, rerank outcome candidates merged with observed outcome at the end.

    Assumptions (matches this notebook):
    - `adr_tst.inst[i]` is the query/case text
    - `adr_trn.pt[i]` is the observed ADR pt (JSON with key 'pt')    
    - `adr_df` has columns: caseid (str), pt (JSON with key 'pt')    
    """
    # Map id -> outcome json
    adr_id_to_outcome = dict(zip(adr_trn.caseid.astype(str), adr_trn.pt))
    
    # generate 5 random outcomes from adr training set for each selected oot inst with 
    # index defined in sample_idx as oot index
    random_outcome_list =[]
    for i in sample_idx:
        random.seed(i)
        random_outcome_list.append(random.sample(adr_trn.pt.tolist(), 5))

    rows = []
    for name in tqdm(model_names, desc="Models", unit="model"):
        model = SentenceTransformer(name, device=device)
        ranks = []
        hit1 = hit3 = hit5 = 0
        mrr5 = ndcg5 = 0.0

        for idx, spl_id in enumerate(sample_idx):
            query = str(adr_tst.inst.iloc[spl_id]) #get oot instruction
            observed = str(adr_tst.pt.iloc[spl_id]) # get oot observed outcome (JSON with key 'pt')           
            soft_neg_outcomes = random_outcome_list[idx] # get 5 random outcomes from adr training set for each selected oot inst with index defined in sample_idx as oot index

            r = _rank_observed_in_candidates(model, query, observed, soft_neg_outcomes, k=5)
            ranks.append(r["rank"])
            hit1 += r["hit@1"]
            hit3 += r["hit@3"]
            hit5 += r["hit@5"]
            mrr5 += r["mrr@5"]
            ndcg5 += r["ndcg@5"]

        ranks_arr = np.array(ranks, dtype=np.int32)
        rows.append(
            {
                "model": name,
                #"n": len(sample_idx),
                "mean_rank": float(ranks_arr.mean()),
                #"median_rank": float(np.median(ranks_arr)),
                "hit@1": hit1 / len(sample_idx),
                "hit@3": hit3 / len(sample_idx),
                "hit@5": hit5 / len(sample_idx),
                "mrr@5": mrr5 / len(sample_idx),
                "ndcg@5": ndcg5 / len(sample_idx),
            }
        )
    
    # clear VRAM
    del model
    gc.collect() 
    torch.cuda.empty_cache()

    return pd.DataFrame(rows).sort_values(by=['mean_rank'], ascending=False)


In [ ]:
# nDCG@5, MRR@5, MAP, Recall, Precision
# Embedding-model reranking (rank the observed outcome high)
# Goal: for each OOT case, merge the retrieved candidate ADR outcomes with the **observed** ADR outcome, score each candidate by similarity to the case `inst` using several embedding models, then compare models by how highly they rank the observed outcome.
# mdl = SentenceTransformer('./bge_m3_triplet', device='cuda')

# _rank_observed_in_candidates(mdl, oot.inst[0], observed_outcome,candidate_outcome, k=5)
# #mdl_ = SentenceTransformer('./bge_m3_4adr', device='cuda')
# #_rank_observed_in_candidates(mdl_, oot.inst[0], observed_outcome,candidate_outcome, k=5)
# torch.cuda.empty_cache()

# import random
# # Example: compare a few embedding models (replace/add yours)
# NOTE: running on all ~117k queries is expensive; start with max_queries=500-2000.
# import torch


In [90]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model_names = [
    "./hybrid_rag/bge_m3_triplet",          # our current fine-tuned model (local)
    "./hybrid_rag/bge_m3",
    "./hybrid_rag/medembed_large",
    "./hybrid_rag/medembed_base",
    "./hybrid_rag/medembed_small",                   
]

In [10]:
(adr_trn, adr_tst) = pickle.load(open("./hybrid_rag/adr_trn_tst_627.pkl", "rb"))

In [91]:
# get random sample of oot indices for quick testing
sample_size = 3000
np.random.seed(1234)
sample_index = np.random.choice(len(adr_tst), size=sample_size, replace=False)
len(sample_index)

3000

In [92]:
%%time 

embed_mdl_summary = evaluate_embed_models(
    model_names=model_names,
    adr_tst=adr_tst,
    adr_trn=adr_trn,    
    sample_idx=sample_index.tolist(),
    device=device,
)

Models:   0%|          | 0/5 [00:00<?, ?model/s]

CPU times: user 31min 53s, sys: 32.2 s, total: 32min 25s
Wall time: 31min 53s


In [95]:
embed_mdl_summary.sort_values(by=['mean_rank'], ascending=True)

,model,mean_rank,hit@1,hit@3,hit@5,mrr@5,ndcg@5
0,./hybrid_rag/bge_m3_triplet,1.314667,0.792667,0.974000,0.997667,0.881583,0.911215
2,./hybrid_rag/medembed_large,2.997667,0.282333,0.612667,0.891667,0.484422,0.584525
4,./hybrid_rag/medembed_small,3.001000,0.266667,0.626667,0.885000,0.477722,0.578138
3,./hybrid_rag/medembed_base,3.039667,0.284667,0.596000,0.876333,0.481450,0.578455
1,./hybrid_rag/bge_m3,3.203000,0.250333,0.559667,0.865333,0.447972,0.550205


In [3]:
#pickle.dump(embed_mdl_summary, open("embed_soc_mdl_summary.pkl", "wb"))
embed_mdl_summary = pickle.load(open("embed_soc_mdl_summary.pkl", "rb"))

In [6]:
embed_mdl_summary.model = ["bge_m3_base", "medembed_base", "medembed_small", "medembed_large", "bge_m3_triplet"]

In [8]:
embed_mdl_summary.sort_values(by=['mean_rank'], ascending=True)

,model,mean_rank,hit@1,hit@3,hit@5,mrr@5,ndcg@5
0,bge_m3_triplet,1.314667,0.792667,0.974000,0.997667,0.881583,0.911215
2,medembed_large,2.997667,0.282333,0.612667,0.891667,0.484422,0.584525
4,medembed_small,3.001000,0.266667,0.626667,0.885000,0.477722,0.578138
3,medembed_base,3.039667,0.284667,0.596000,0.876333,0.481450,0.578455
1,bge_m3_base,3.203000,0.250333,0.559667,0.865333,0.447972,0.550205


# compare data w/o normalized drug name and ATC class

In [96]:
model_names_ = [
    "./hybrid_rag/bge_m3_4adr",          # our current fine-tuned model (local)
    "./hybrid_rag/bge_m3",
    "./hybrid_rag/medembed_large",
    "./hybrid_rag/medembed_base",
    "./hybrid_rag/medembed_small",                   
]

In [97]:
%%time 

embed_mdl_summary_ = evaluate_embed_models(
    model_names=model_names_,
    adr_tst=adr_tst,
    adr_trn=adr_trn,    
    sample_idx=sample_index.tolist(),
    device=device,
)

Models:   0%|          | 0/5 [00:00<?, ?model/s]

CPU times: user 32min 52s, sys: 39 s, total: 33min 31s
Wall time: 32min 53s


In [98]:
# compare bge_m3_4adr w/o normalized drug name, ATC4 class, and yr_qtr 
embed_mdl_summary_.sort_values(by=['mean_rank'], ascending=True)

,model,mean_rank,hit@1,hit@3,hit@5,mrr@5,ndcg@5
0,./hybrid_rag/bge_m3_4adr,1.656000,0.662000,0.910667,0.985000,0.789572,0.838738
2,./hybrid_rag/medembed_large,2.997667,0.282333,0.612667,0.891667,0.484422,0.584525
4,./hybrid_rag/medembed_small,3.001000,0.266667,0.626667,0.885000,0.477722,0.578138
3,./hybrid_rag/medembed_base,3.039667,0.284667,0.596000,0.876333,0.481450,0.578455
1,./hybrid_rag/bge_m3,3.203000,0.250333,0.559667,0.865333,0.447972,0.550205


In [99]:
pickle.dump(embed_mdl_summary_, open("embed_soc_no_norm_class_summary.pkl", "wb"))

In [ ]:
# save adr_trn and adr_tst
#pickle.dump((adr_trn, adr_tst), open("adr_trn_tst.pkl", "wb"))

In [ ]:
# calculate PPR, ROR, and EBGM scores for each case in adr_tst 
# using the training set adr_trn as reference.
from scipy.special import xlogy

def calculate_ebgm(a, b, c, d, alpha0=1, beta0=1):
    """
    Calculate Empirical Bayes Geometric Mean
    
    Args:
        a: drug + event
        b: drug, no event  
        c: no drug + event
        d: no drug, no event
        alpha0, beta0: hyperparameters (default: 1)
    
    Returns:
        ebgm: Empirical Bayes Geometric Mean
    """
    N = a + b + c + d
    
    # Expected count under independence
    E_a = ((a + b) * (a + c)) / N
    
    # Empirical Bayes posterior estimate
    a_hat = (beta0 * alpha0 + a) / (beta0 + 1)
    
    # O/E ratio
    O_E = a / a_hat if a_hat > 0 else 0
    
    # EBGM (geometric mean)
    ebgm = np.exp(np.sqrt(np.log(O_E))) if O_E > 0 else 0
    
    return ebgm

# Example
ebgm = calculate_ebgm(a=50, b=950, c=100, d=9900)
print(f"EBGM: {ebgm:.3f}")

In [45]:
# load adr trn  tst splits
(adr_trn, adr_tst) = pickle.load(open("adr_trn_tst.pkl", "rb"))

In [23]:
ls adr_trn_tst.pkl -al ## 5038297116

-rw-rw-r-- 1 dada dada 5038297105 Jun 26 22:08 adr_trn_tst.pkl


In [ ]:
# load HLT
# import LLT
hlt_file = "MedDRA/MedDRA_Releases/MedDRA_29_0_English/MedAscii/hlt.asc"
hlt_df = pd.read_csv(hlt_file, sep=r"\$", header=None, engine="pyth

In [348]:
hlt_df = hlt_df.iloc[:,[0,1]]

In [349]:
mdhier_df.head()

,pt_code,hlt_code,hlgt_code,soc_code,pt_name,hlt_name,hlgt_name,soc_name,soc_abbrev,pt_soc_code,primary_soc_fg
0,10002043,10002042,10002086,10005329,anaemia folate deficiency,anaemia deficiencies,anaemias nonhaemolytic and marrow depression,blood and lymphatic system disorders,blood,10005329,y
1,10002080,10002042,10002086,10005329,anaemia vitamin b12 deficiency,anaemia deficiencies,anaemias nonhaemolytic and marrow depression,blood and lymphatic system disorders,blood,10005329,y
2,10002081,10002042,10002086,10005329,anaemia vitamin b6 deficiency,anaemia deficiencies,anaemias nonhaemolytic and marrow depression,blood and lymphatic system disorders,blood,10005329,y
3,10022972,10002042,10002086,10005329,iron deficiency anaemia,anaemia deficiencies,anaemias nonhaemolytic and marrow depression,blood and lymphatic system disorders,blood,10005329,y
4,10034695,10002042,10002086,10005329,pernicious anaemia,anaemia deficiencies,anaemias nonhaemolytic and marrow depression,blood and lymphatic system disorders,blood,10005329,y


In [ ]:
list(mdhier_df.hlgt_name)[:5]

['anaemia deficiencies',
 'anaemia deficiencies',
 'anaemia deficiencies',
 'anaemia deficiencies',
 'anaemia deficiencies']

In [360]:
in_hlgt = [i for i in pt_out if i in list(mdhier_df.hlgt_name.unique())]
in_hlt = [i for i in pt_out if i in list(mdhier_df.hlt_name.unique())]

In [359]:
len(in_hlt)

0

## Embedding-model reranking (rank the observed outcome high)

Goal: for each OOT case, merge the retrieved candidate ADR outcomes with the **observed** ADR outcome, score each candidate by similarity to the case `inst` using several embedding models, then compare models by how highly they rank the observed outcome.

In [ ]:
observed_outcome = oot.outcome[0]
candidate_outcome = [adr.loc[adr.id == caseid].outcome.values[0] for caseid in bm25_ids[0]]
fused_pts = candidate_outcome + [observed_outcome]

In [ ]:
mdl = SentenceTransformer('./medembed_large', device='cuda')

In [ ]:
_rank_observed_in_candidates(mdl, oot.inst[0], observed_outcome,candidate_outcome, k=5)

In [ ]:
#mdl_ = SentenceTransformer('./bge_m3_4adr', device='cuda')

In [ ]:
#_rank_observed_in_candidates(mdl_, oot.inst[0], observed_outcome,candidate_outcome, k=5)

In [ ]:
torch.cuda.empty_cache()

In [ ]:
def _rank_observed_in_candidates(
    model: SentenceTransformer,
    query_inst: str,
    observed_outcome: str,
    adr_random_outcomes: list[str],
    k: int = 5,
) -> dict:
    """
    Returns rank (1-based) of observed among candidates, plus top-k list.

    Scoring: cosine(sim(emb(query_inst), emb(candidate_outcome_pt))).
    """
    # Candidates = retrieved outcomes + observed outcome (ensure included)
    fused_pts = adr_random_outcomes + [observed_outcome] #5+1

    # Keep first occurrence order but unique (important if observed already present)
    seen = set()
    uniq_cand_pts = []
    for c in fused_pts:
        if c not in seen:
            uniq_cand_pts.append(c)
            seen.add(c)

    q_emb = model.encode([query_inst], normalize_embeddings=True)
    d_embs = model.encode(uniq_cand_pts, normalize_embeddings=True)

    # get cosine similarity and rank
    sims = (q_emb @ d_embs.T).ravel().astype(float)
    order = np.argsort(-sims)
    ranked = [(uniq_cand_pts[i], sims[i]) for i in order]

    # rank of observed is 1-based index of observed in ranked list
    obs_idx = next(i for i, (txt, _) in enumerate(ranked) if txt == fused_pts[-1])
    rank = obs_idx + 1

    return {
        "rank": rank,
        "hit@1": 1 if rank <= 1 else 0,
        "hit@3": 1 if rank <= 3 else 0,
        "hit@5": 1 if rank <= 5 else 0,
        "mrr@5": (1.0 / rank) if rank <= 5 else 0.0,
        # single-relevant-item nDCG@5
        "ndcg@5": float((1.0 / np.log2(rank + 1)) if rank <= 5 else 0.0),
        "topk": ranked[:k],
    }


def evaluate_embed_models(
    model_names: list[str],
    oot_df: pd.DataFrame,
    adr_df: pd.DataFrame,         
    sample_idx: list[int],
    device: str = "cuda",
    random_seed = 1234
) -> pd.DataFrame:
    """
    For each model, rerank outcome candidates merged with observed outcome at the end.

    Assumptions (matches this notebook):
    - `oot_df.inst[i]` is the query/case text
    - `oot_df.outcome[i]` is the observed ADR outcome (JSON with key 'pt')    
    - `adr_df` has columns: id (str), outcome (JSON with key 'pt')
    - `fuse_ids[i]` is an iterable of ADR ids retrieved for query i
    """
    # Map id -> outcome json
    adr_id_to_outcome = dict(zip(adr_df.id.astype(str), adr_df.outcome))
    
    # generate 5 random outcomes from adr training set for each selected oot inst with 
    # index defined in sample_idx as oot index
    random_outcome_list =[]
    for i in sample_idx:
        random.seed(i)
        random_outcome_list.append(random.sample(adr_df.outcome.tolist(), 5))

    rows = []
    for name in tqdm(model_names):
        model = SentenceTransformer(name, device=device)
        ranks = []
        hit1 = hit3 = hit5 = 0
        mrr5 = ndcg5 = 0.0

        for idx, spl_id in enumerate(sample_idx):
            query = str(oot_df.inst.iloc[spl_id]) #get oot instruction
            observed = str(oot_df.outcome.iloc[spl_id]) # get oot observed outcome (JSON with key 'pt')           
            soft_neg_outcomes = random_outcome_list[idx] # get 5 random outcomes from adr training set for each selected oot inst with index defined in sample_idx as oot index

            r = _rank_observed_in_candidates(model, query, observed, soft_neg_outcomes, k=5)
            ranks.append(r["rank"])
            hit1 += r["hit@1"]
            hit3 += r["hit@3"]
            hit5 += r["hit@5"]
            mrr5 += r["mrr@5"]
            ndcg5 += r["ndcg@5"]

        ranks_arr = np.array(ranks, dtype=np.int32)
        rows.append(
            {
                "model": name,
                #"n": len(sample_idx),
                "mean_rank": float(ranks_arr.mean()),
                #"median_rank": float(np.median(ranks_arr)),
                "hit@1": hit1 / len(sample_idx),
                "hit@3": hit3 / len(sample_idx),
                "hit@5": hit5 / len(sample_idx),
                "mrr@5": mrr5 / len(sample_idx),
                "ndcg@5": ndcg5 / len(sample_idx),
            }
        )
    
    # clear VRAM
    del model
    gc.collect() 
    torch.cuda.empty_cache()

    return pd.DataFrame(rows).sort_values(by=['mean_rank'], ascending=False)


In [ ]:
import random

In [ ]:
# Example: compare a few embedding models (replace/add yours)
# NOTE: running on all ~117k queries is expensive; start with max_queries=500-2000.
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model_names = [
    "./bge_m3_4adr",          # our current fine-tuned model (local)
    "./bge_m3",
    "./medembed_large",
    "./medembed_base",
    "./medembed_small",                   
]

In [ ]:
# get random sample of oot indices for quick testing
sample_size = 3000
np.random.seed(1234)
sample_index = np.random.choice(len(oot), size=sample_size, replace=False)
len(sample_index)

In [ ]:
embed_mdl_summary = evaluate_embed_models(
    model_names=model_names,
    oot_df=oot,
    adr_df=adr,    
    sample_idx=sample_index.tolist(),
    device=device,
)